# TabNetClassifier RNA-seq Classification with Cross-Validation + fold-wise pyDESeq2 feature selection

이 노트북은 PPMI RNA-seq TPM 데이터를 이용해 **PD vs Control 분류**를 수행하는 실험용 노트북입니다.

주요 기능은 다음과 같습니다.

1. RNA-seq TPM matrix와 metadata 병합
2. TabNet 학습용 데이터셋 구성
3. 환자 단위 group 기반 교차 검증
4. 선택적으로 각 fold의 **train set에서만 pyDESeq2 차등발현분석(DEG)을 수행**하고, 선별된 gene만 모델 feature로 사용
5. 이미 저장된 fold별 DESeq 결과 CSV 재사용
6. TabNet 모델 학습 및 fold별 결과 저장
7. 선택적 baseline 모델(SVM, Random Forest, XGBoost 등) 학습 및 결과 저장
8. 평가 지표, confusion matrix, classification report, feature importance, 학습 곡선 저장

> 중요: leakage 방지를 위해 DESeq feature selection은 반드시 fold별 train split에서만 수행됩니다. validation split의 label은 DESeq feature selection에 사용되지 않습니다.


In [ ]:
# 1. 라이브러리 import 및 환경 확인


import os
import gc
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.pretraining import TabNetPretrainer

print("PyTorch version:", torch.__version__)

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ['CUDA_VISIBLE_DEVICES'] = '3'

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Available device:", device)


## 2. RNA Sequencing TPM 로딩




In [ ]:
def normalize_diagnosis_for_tabnet(x):
    """
    PPMI metadata의 진단명을 TabNet 실험용 표준 그룹명으로 변환합니다.

    예:
    - "PD" -> "PD"
    - "Control", "HC", "Healthy Control" -> "Control"
    - "Prodromal" -> "Prodromal"
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    mapping = {
        "PD": "PD",
        "Control": "Control",
        "HC": "Control",
        "Healthy control": "Control",
        "Healthy Control": "Control",
        "Prodromal": "Prodromal",
        "SWEDD": "SWEDD",
        "Genetic Cohort": "Genetic Cohort",
        "Genetic Registry": "Genetic Registry",
        "PDPOOL": "PDPOOL",
        "HCPOOL": "Control",
    }

    return mapping.get(x, x)


def normalize_visit_for_filter(x):
    """
    방문시점 값을 비교하기 쉽게 정규화합니다.

    예:
    - "BL" -> "BL"
    - "Baseline" -> "BASELINE"
    - " V04 " -> "V04"
    """
    if pd.isna(x):
        return np.nan
    return str(x).strip()


def get_gene_feature_cols(df):
    """
    TPM matrix에서 유전자 feature column을 추출합니다.
    현재 노트북 방식과 동일하게 ENSG 또는 ENST로 시작하는 column을 사용합니다.
    """
    feature_cols = []
    for c in df.columns:
        if re.match(r"^(ENSG|ENST)\d+", str(c)):
            feature_cols.append(c)
    return feature_cols

In [ ]:
# 2. RNA-seq TPM matrix와 metadata 로딩

RANDOM_STATE = 42

tpm_df = pd.read_csv('./Dataset/RNA/IR3/quant/tpm_gene_matrix.csv', index_col=0)
meta_df = pd.read_csv('./Dataset/RNA/IR3/metaDataIR3.csv', index_col=0)

print("Dataset shape:", tpm_df.shape)
display(tpm_df.head())
display(meta_df.head())


In [ ]:
def build_tabnet_rna_classification_dataset(
    merged_df,
    selected_groups=("PD", "Control"),
    # visit_values=("BL", "Baseline"),
    visit_values=None,
    visit_col="auto",
    diagnosis_col="auto",
    patient_col="PATNO_final",
    feature_cols=None,
    test_size=0.2,
    random_state=42,
    log1p_tpm=True,
    scale_features=False,
    fillna_value=0.0,
    drop_qc_fail=True,
    qc_col="QCflagIR3",
    qc_pass_value="pass",
    label_map=None,
    verbose=True,
):
    """
    PPMI RNA-seq TPM 데이터를 TabNetClassifier 학습용으로 구성합니다.

    Parameters
    ----------
    merged_df : pd.DataFrame
        TPM + metadata가 병합된 DataFrame.
        현재 노트북의 merged_df를 그대로 넣으면 됩니다.

    selected_groups : tuple or list
        사용할 진단군.
        기본값: ("PD", "Control")

    visit_values : tuple, list, str, or None
        사용할 방문시점.
        기본값: ("BL", "Baseline")
        None이면 방문시점 필터링을 하지 않습니다.

    visit_col : str
        방문시점 기준 column.
        "auto"이면 VISIT_final > CLINICAL_EVENT > VISIT 순서로 자동 선택합니다.

    diagnosis_col : str
        진단명 기준 column.
        "auto"이면 DIAGNOSIS, DIAGONOSIS, DIAGNOSIS_STD 등을 자동 탐색합니다.

    patient_col : str
        환자 ID column.
        기본값은 현재 노트북의 PATNO_final입니다.

    feature_cols : list or None
        사용할 유전자 feature column.
        None이면 ENSG/ENST로 시작하는 column을 자동 선택합니다.

    log1p_tpm : bool
        TPM 값에 np.log1p 변환을 적용할지 여부.
        RNA-seq TPM은 skew가 크기 때문에 기본값 True를 권장합니다.

    scale_features : bool
        StandardScaler 적용 여부.
        적용할 경우 train set에만 fit하고 eval set에는 transform만 적용합니다.

    label_map : dict or None
        예: {"Control": 0, "PD": 1}
        None이면 기본적으로 Control=0, PD=1, Prodromal=2 순서로 자동 생성합니다.

    Returns
    -------
    result : dict
        X_train, X_eval, y_train, y_eval, train_df, eval_df, feature_cols, label_map 등을 포함합니다.
    """

    df = merged_df.copy()

    # -------------------------
    # 1. feature column 결정
    # -------------------------
    if feature_cols is None:
        feature_cols = get_gene_feature_cols(df)

    if len(feature_cols) == 0:
        raise ValueError("유전자 feature column을 찾지 못했습니다. ENSG 또는 ENST로 시작하는 column이 있는지 확인하세요.")

    # -------------------------
    # 2. diagnosis column 결정
    # -------------------------
    if diagnosis_col == "auto":
        diagnosis_candidates = [
            "DIAGNOSIS",
            "DIAGONOSIS",      # 사용자가 언급한 오탈자 가능성까지 대응
            "DIAGNOSIS_STD",
            "diagnosis",
            "Diagnosis",
        ]

        diagnosis_col = None
        for c in diagnosis_candidates:
            if c in df.columns:
                diagnosis_col = c
                break

        if diagnosis_col is None:
            raise ValueError(
                "진단명 column을 찾지 못했습니다. "
                "DIAGNOSIS 또는 DIAGONOSIS column이 있는지 확인하세요."
            )

    # -------------------------
    # 3. visit column 결정
    # -------------------------
    if visit_col == "auto":
        visit_candidates = [
            "VISIT_final",
            "CLINICAL_EVENT",
            "VISIT",
            "EVENT_ID",
        ]

        visit_col = None
        for c in visit_candidates:
            if c in df.columns:
                visit_col = c
                break

        if visit_col is None and visit_values is not None:
            raise ValueError(
                "방문시점 column을 찾지 못했습니다. "
                "VISIT_final, CLINICAL_EVENT, VISIT 중 하나가 필요합니다."
            )

    # -------------------------
    # 4. 환자 ID column 확인
    # -------------------------
    if patient_col not in df.columns:
        raise ValueError(f"patient_col='{patient_col}'이 merged_df에 없습니다.")

    # -------------------------
    # 5. 진단명 표준화
    # -------------------------
    df["DIAGNOSIS_GROUP"] = df[diagnosis_col].apply(normalize_diagnosis_for_tabnet)

    # -------------------------
    # 6. 방문시점 필터링
    # -------------------------
    if visit_values is not None:
        if isinstance(visit_values, str):
            visit_values = [visit_values]

        visit_values_norm = [normalize_visit_for_filter(v) for v in visit_values]

        df["_VISIT_FOR_FILTER"] = df[visit_col].apply(normalize_visit_for_filter)
        df = df[df["_VISIT_FOR_FILTER"].isin(visit_values_norm)].copy()

    # -------------------------
    # 7. 진단군 필터링
    # -------------------------
    selected_groups = list(selected_groups)
    df = df[df["DIAGNOSIS_GROUP"].isin(selected_groups)].copy()

    # -------------------------
    # 8. QC pass 필터링
    # -------------------------
    if drop_qc_fail and qc_col in df.columns:
        df = df[df[qc_col] == qc_pass_value].copy()

    # -------------------------
    # 9. 결측 필터링
    # -------------------------
    df = df.dropna(subset=["DIAGNOSIS_GROUP", patient_col]).copy()

    # -------------------------
    # 10. label map 생성
    # -------------------------
    if label_map is None:
        preferred_order = ["Control", "PD", "Prodromal", "SWEDD"]

        ordered_groups = []
        for g in preferred_order:
            if g in selected_groups:
                ordered_groups.append(g)

        for g in selected_groups:
            if g not in ordered_groups:
                ordered_groups.append(g)

        label_map = {g: i for i, g in enumerate(ordered_groups)}

    df["label"] = df["DIAGNOSIS_GROUP"].map(label_map)
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    inverse_label_map = {v: k for k, v in label_map.items()}

    # -------------------------
    # 11. 유전자 발현 matrix 구성
    # -------------------------
    X_df = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(fillna_value)

    if log1p_tpm:
        # TPM은 0 이상이어야 하므로 음수 방어 처리
        X_df = X_df.clip(lower=0)
        X_df = np.log1p(X_df)

        # np.log1p 결과가 ndarray일 수 있으므로 다시 DataFrame 보장
        X_df = pd.DataFrame(X_df, columns=feature_cols, index=df.index)

    y = df["label"]

    # -------------------------
    # 12. 환자 단위 train/eval split
    # -------------------------
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state,
    )

    idx = np.arange(len(df))
    groups = df[patient_col].astype(str)

    train_idx, eval_idx = next(splitter.split(idx, y, groups=groups))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    eval_df = df.iloc[eval_idx].reset_index(drop=True)

    X_train_df = X_df.iloc[train_idx].reset_index(drop=True)
    X_eval_df = X_df.iloc[eval_idx].reset_index(drop=True)

    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_eval = y.iloc[eval_idx].reset_index(drop=True)

    # -------------------------
    # 13. 선택적 scaling
    # -------------------------
    scaler = None

    if scale_features:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_df).astype(np.float32)
        X_eval = scaler.transform(X_eval_df).astype(np.float32)
    else:
        X_train = X_train_df.to_numpy(dtype=np.float32)
        X_eval = X_eval_df.to_numpy(dtype=np.float32)

    y_train = y_train.to_numpy(dtype=np.int64)
    y_eval = y_eval.to_numpy(dtype=np.int64)

    # -------------------------
    # 14. 출력 확인
    # -------------------------
    if verbose:
        print("=" * 80)
        print("[TabNet RNA dataset]")
        print("diagnosis_col:", diagnosis_col)
        print("visit_col:", visit_col)
        print("selected_groups:", selected_groups)
        print("visit_values:", visit_values)
        print("feature count:", len(feature_cols))
        print("total samples:", len(df))
        print("train samples:", len(train_df))
        print("eval samples:", len(eval_df))
        print("\n[label_map]")
        print(label_map)

        print("\n[전체 class 분포]")
        print(df["DIAGNOSIS_GROUP"].value_counts())

        print("\n[Train class 분포]")
        print(train_df["DIAGNOSIS_GROUP"].value_counts())

        print("\n[Eval class 분포]")
        print(eval_df["DIAGNOSIS_GROUP"].value_counts())

        print("\n[환자 수]")
        print("total patients:", df[patient_col].nunique())
        print("train patients:", train_df[patient_col].nunique())
        print("eval patients:", eval_df[patient_col].nunique())

        overlap_patients = set(train_df[patient_col].astype(str)) & set(eval_df[patient_col].astype(str))
        print("patient overlap between train/eval:", len(overlap_patients))
        print("=" * 80)

    # 교차검증용 전체 데이터도 함께 반환합니다.
    # X_all_df는 log1p/fillna까지 적용된 값이며, fold별 scaling이 필요하면
    # 교차검증 루프 안에서 train fold에만 fit하도록 처리합니다.
    X_all_df = X_df.reset_index(drop=True)
    y_all = y.reset_index(drop=True).to_numpy(dtype=np.int64)
    filtered_df = df.reset_index(drop=True)
    groups_all = filtered_df[patient_col].astype(str).reset_index(drop=True).to_numpy()

    return {
        "X_train": X_train,
        "X_eval": X_eval,
        "y_train": y_train,
        "y_eval": y_eval,
        "X_train_df": X_train_df,
        "X_eval_df": X_eval_df,
        "train_df": train_df,
        "eval_df": eval_df,
        "X_all": X_all_df.to_numpy(dtype=np.float32),
        "X_all_df": X_all_df,
        "y_all": y_all,
        "filtered_df": filtered_df,
        "groups": groups_all,
        "patient_col": patient_col,
        "feature_cols": feature_cols,
        "label_map": label_map,
        "inverse_label_map": inverse_label_map,
        "scaler": scaler,
        "diagnosis_col": diagnosis_col,
        "visit_col": visit_col,
    }

In [ ]:
def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if re.fullmatch(r"\d+(\.0)?", x):
        x = x.replace(".0", "")
        x = x.lstrip("0")
        return x if x != "" else "0"

    return x


def normalize_patno(x):
    if pd.isna(x):
        return np.nan

    try:
        return str(int(float(x)))
    except:
        return str(x).strip()


def normalize_visit(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()


def extract_hudalphaid_or_nan(x):
    """
    metadata 또는 파일명 문자열에서 featureCounts 파일명과 매칭되는
    5104-SL-... 형식의 HudAlphaID를 추출합니다.
    찾지 못하면 np.nan을 반환합니다.
    """
    if pd.isna(x):
        return np.nan

    m = re.search(r"5104-SL-\d+", str(x))
    if m is None:
        return np.nan
    return m.group(0)


def ensure_hudalphaid_column(meta_df, target_col="HudAlphaID"):
    """
    task1_2_RNA_CV_v2의 pyDESeq2 실행 방식에 맞추어,
    metadata에 featureCounts raw count matrix와 매칭 가능한 HudAlphaID 컬럼을 보장합니다.

    우선순위:
    1. 이미 존재하는 HudAlphaID 컬럼에서 5104-SL-... 추출
    2. meta_df index에서 5104-SL-... 추출
    3. Specimen Bar Code, SAMPLE_ID 등 후보 컬럼에서 5104-SL-... 추출

    반환값은 HudAlphaID 컬럼이 추가/정규화된 meta_df입니다.
    """
    out = meta_df.copy()

    candidate_sources = []

    if target_col in out.columns:
        candidate_sources.append((target_col, out[target_col]))

    # read_csv(..., index_col=0) 때문에 HudAlphaID가 index로 들어간 경우까지 대응합니다.
    candidate_sources.append(("index", pd.Series(out.index, index=out.index)))

    preferred_cols = [
        "Specimen Bar Code",
        "SAMPLE_ID",
        "Sample ID",
        "sample_id",
        "HudAlphaID_meta",
        "File Name",
        "filename",
    ]
    for c in preferred_cols:
        if c in out.columns and c != target_col:
            candidate_sources.append((c, out[c]))

    # 이름상 sample/hud 관련 컬럼도 추가 탐색합니다.
    for c in out.columns:
        low = str(c).lower()
        if any(k in low for k in ["hud", "alpha", "sample", "specimen", "barcode", "bar code"]):
            if c != target_col and c not in preferred_cols:
                candidate_sources.append((c, out[c]))

    best_name = None
    best_values = None
    best_n = -1

    for name, values in candidate_sources:
        extracted = values.apply(extract_hudalphaid_or_nan)
        n = int(extracted.notna().sum())
        print(f"HudAlphaID 후보 {name}: 추출 가능 수 = {n}")
        if n > best_n:
            best_name = name
            best_values = extracted
            best_n = n

    if best_n > 0:
        out[target_col] = best_values
        print(f"HudAlphaID를 '{best_name}'에서 추출/정규화했습니다. unique={out[target_col].nunique(dropna=True)}")
    else:
        # 여기서 바로 중단하지는 않습니다. DESeq를 쓰지 않는 실행도 가능하기 때문입니다.
        # 다만 USE_DESEQ_FEATURE_SELECTION=True이면 이후 검증 단계에서 명확한 오류가 납니다.
        print("경고: metadata에서 5104-SL-... 형식의 HudAlphaID를 찾지 못했습니다.")

    return out


In [ ]:
tpm_df = tpm_df.copy()
meta_df = meta_df.copy()

# TPM
tpm_df["PATNO_norm"] = tpm_df["PATNO"].apply(normalize_patno) if "PATNO" in tpm_df.columns else np.nan
tpm_df["VISIT_norm"] = tpm_df["VISIT"].apply(normalize_visit) if "VISIT" in tpm_df.columns else np.nan
tpm_df["SAMPLE_ID_norm"] = tpm_df["SAMPLE_ID"].apply(normalize_sample_id) if "SAMPLE_ID" in tpm_df.columns else np.nan

# metaData
meta_df["PATNO_norm"] = meta_df["PATNO"].apply(normalize_patno) if "PATNO" in meta_df.columns else np.nan
meta_df["VISIT_norm"] = meta_df["CLINICAL_EVENT"].apply(normalize_visit) if "CLINICAL_EVENT" in meta_df.columns else np.nan
meta_df["SAMPLE_ID_norm"] = meta_df["Specimen Bar Code"].apply(normalize_sample_id) if "Specimen Bar Code" in meta_df.columns else np.nan

# pyDESeq2 raw count matrix와 매칭할 HudAlphaID를 metadata에 보장합니다.
meta_df = ensure_hudalphaid_column(meta_df, target_col="HudAlphaID")

print(tpm_df[["PATNO", "VISIT", "SAMPLE_ID", "SAMPLE_ID_norm"]].head())
print(meta_df[["PATNO", "CLINICAL_EVENT", "Specimen Bar Code", "SAMPLE_ID_norm"]].head())

print("HudAlphaID in meta_df:", "HudAlphaID" in meta_df.columns)
if "HudAlphaID" in meta_df.columns:
    print("HudAlphaID unique:", meta_df["HudAlphaID"].nunique(dropna=True))
    print("HudAlphaID 예시:", meta_df["HudAlphaID"].dropna().astype(str).head().tolist())


In [ ]:
# metadata에서 TPM matrix와 병합할 컬럼을 유지합니다.
# SAMPLE_ID_norm은 TPM-meta 병합용 key이고, HudAlphaID는 pyDESeq2 raw count matrix 매칭용 key입니다.
meta_keep_cols = [
    "SAMPLE_ID_norm",
    "PATNO_norm",
    "VISIT_norm",
    "HudAlphaID",
    "QCflagIR3",
    "DIAGNOSIS",
    "GENDER",
    "CLINICAL_EVENT"
]

missing_meta_keep_cols = [c for c in meta_keep_cols if c not in meta_df.columns]
if missing_meta_keep_cols:
    print("metadata에서 찾지 못한 선택 컬럼:", missing_meta_keep_cols)

meta_keep_cols = [c for c in meta_keep_cols if c in meta_df.columns]
meta_sub = meta_df[meta_keep_cols].copy()

if "HudAlphaID" not in meta_sub.columns:
    raise ValueError(
        "meta_sub에 HudAlphaID가 없습니다. "
        "pyDESeq2 feature selection을 사용하려면 metadata에서 5104-SL-... 형식의 HudAlphaID를 복구해야 합니다."
    )

merged_df = pd.merge(
    tpm_df,
    meta_sub,
    on="SAMPLE_ID_norm",
    how="left",
    suffixes=("", "_meta")
)

print("after TPM + meta merge:", merged_df.shape)
print("DIAGNOSIS 매칭 비율:", merged_df["DIAGNOSIS"].notna().mean())
print("HudAlphaID 매칭 비율:", merged_df["HudAlphaID"].notna().mean())
print("HudAlphaID unique:", merged_df["HudAlphaID"].nunique(dropna=True))
print("HudAlphaID 예시:", merged_df["HudAlphaID"].dropna().astype(str).head().tolist())


In [ ]:
def get_feature_cols(df):
    feature_cols = []
    for c in df.columns:
        if re.match(r"^(ENSG|ENST)\d+", str(c)):
            feature_cols.append(c)
    return feature_cols


feature_cols = get_feature_cols(merged_df)
print("feature 개수:", len(feature_cols))
print("예시:", feature_cols[:5])

In [ ]:
tabnet_data = build_tabnet_rna_classification_dataset(
    merged_df=merged_df,
    selected_groups=("PD", "Control"),
    # visit_values=("BL", "Baseline"),
    visit_col="auto",
    diagnosis_col="auto",
    patient_col="PATNO_norm",
    feature_cols=feature_cols,
    test_size=0.2,
    random_state=42,
    log1p_tpm=True,
    scale_features=False,
)

## 3. 실험 설정 및 결과 저장 디렉토리

아래 셀에서 결과 저장 위치, fold 수, TabNet 학습 파라미터, baseline 모델 사용 여부를 지정합니다.

- `RESULTS_DIR`: 결과를 저장할 최상위 디렉토리
- `RUN_NAME`: 각 실행을 구분하는 이름
- `N_SPLITS`: 교차검증 fold 수
- `BASELINE_MODELS`: 기본값은 `None`이며, 필요 시 `["svm_linear", "rf", "xgboost"]`처럼 지정합니다.


In [ ]:
# 3. 실험 설정

from pathlib import Path
from datetime import datetime
import json
import joblib
import warnings

from sklearn.model_selection import GroupKFold, StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

RESULTS_DIR = Path("./results_tabnet_rna11")
RUN_NAME = f"pd_control_tabnet_cv_ssl_finetune_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RESULTS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 교차검증 fold 수
N_SPLITS = 5

# TabNet 교차검증 실행 여부
RUN_TABNET_CV = True

# Baseline 모델 실행 여부
# 기본값: None 또는 []이면 baseline을 실행하지 않습니다.
# 사용 예시:
# BASELINE_MODELS = ["svm_linear", "rf"]
# BASELINE_MODELS = ["svm_linear", "rf", "xgboost"]
# BASELINE_MODELS = "all"
BASELINE_MODELS = None


# =========================
# TabNet scratch / pretrained fine-tuning 실행 설정
# =========================
# "scratch": 기존 방식처럼 TabNetClassifier를 처음부터 지도학습합니다.
# "pretrained": fold별로 저장된 TabNetPretrainer를 불러와 from_unsupervised로 fine-tuning합니다.
# "both": scratch와 pretrained fine-tuning을 모두 실행하여 같은 CV split에서 비교합니다.
# 기본값은 기존 노트북과 동일하게 scratch만 실행되도록 둡니다.
TABNET_TRAIN_MODE = "scratch"  # "scratch", "pretrained", "both"

# 사전학습 전용 스크립트가 저장한 전체 run directory를 지정합니다.
# 예: Path("./results_tabnet_ssl_pretrain_only/ppmi_tabnet_ssl_pretrain_20260615_123456")
# 내부에는 fold_01, fold_02, ... 디렉토리가 있어야 합니다.
PRETRAINED_TABNET_RUN_DIR = './results_tabnet/ppmi_tabnet_ssl_pretrain_only_20260615_184608'
PRETRAINED_MODEL_FILENAME = "tabnet_pretrainer_full.zip"
PRETRAINED_FEATURE_FILENAME = "selected_feature_columns.csv"
PRETRAINED_SCALER_FILENAME = "scaler.joblib"

# pretrained fine-tuning에서는 Encoder 입력 차원/순서를 맞추기 위해
# 사전학습 때 저장된 selected_feature_columns.csv를 기본적으로 사용합니다.
USE_PRETRAINED_FEATURES = True
STRICT_PRETRAINED_FEATURE_MATCH = True

# 사전학습 때 scale_within_fold=True였다면 fold별 scaler.joblib를 함께 사용하는 편이 안전합니다.
USE_PRETRAINED_SCALER = True

# pretrained fine-tuning 시 Encoder를 freeze할지 여부입니다.
# False 기본값: from_unsupervised로 사전학습 가중치를 초기화한 뒤 Encoder까지 함께 업데이트합니다.
# True: pretrained Encoder는 고정하고, downstream classification head 중심으로 학습합니다.
FREEZE_PRETRAINED_ENCODER = True

# categorical embedding이 있는 경우 embedder까지 Encoder 쪽 표현부로 보고 함께 freeze할지 여부입니다.
# RNA-seq TPM 입력에서는 보통 categorical embedding이 없지만, 구조 호환성을 위해 옵션으로 둡니다.
FREEZE_PRETRAINED_EMBEDDER = True

# 학습 시작 시 Encoder/Head의 trainable parameter 수를 출력합니다.
PRINT_PRETRAINED_ENCODER_TRAINABILITY = True

# fold별 feature scaling 여부
# TabNet은 False를 기본으로 두고, SVM/LogisticRegression baseline은 Pipeline 내부에서 scaling합니다.
SCALE_TABNET_WITHIN_FOLD = False

TABNET_MODEL_PARAMS = dict(
    n_d=256,
    n_a=256,
    n_steps=4,
    gamma=1.3,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-4),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=20, gamma=0.8),
    mask_type="sparsemax",
    verbose=1,
)

TABNET_FIT_PARAMS = dict(
    eval_metric=["accuracy", "balanced_accuracy"],
    max_epochs=200,
    patience=40,
    batch_size=64,
    virtual_batch_size=16,
    num_workers=0,
    drop_last=False,
)

# =========================
# pyDESeq2 기반 DEG feature selection 설정
# =========================
# True이면 각 fold의 train split에서만 pyDESeq2를 수행하거나 기존 fold별 결과를 불러온 뒤,
# 조건을 만족하는 DEG만 모델 입력 feature로 사용합니다.

USE_DESEQ_FEATURE_SELECTION = False  # backward compatibility용 전역 기본값. 아래 scope 설정이 우선합니다.

# 모델별 DESeq feature selection 적용 범위입니다.
# "none": 어떤 모델에도 DESeq feature selection을 적용하지 않음. 기본값.
# "all": 모든 모델에 적용.
# "tabnet": tabnet_scratch와 tabnet_pretrained_finetune에 적용.
# "baselines": TabNet을 제외한 baseline 모델에만 적용.
# "custom": DESEQ_FEATURE_SELECTION_MODELS에 명시한 모델에만 적용.
DESEQ_FEATURE_SELECTION_SCOPE = "none"  # "none", "all", "tabnet", "baselines", "custom"
DESEQ_FEATURE_SELECTION_MODELS = []      # 예: ["tabnet_scratch", "svm_linear", "rf"]


# featureCounts raw count 파일 위치입니다.
# pyDESeq2에는 TPM이 아니라 raw integer count를 넣어야 합니다.
COUNTS_DIR = Path("./Dataset/RNA/IR3/counts/")

# DESeq 결과 저장 위치입니다. RUN_DIR 내부에 fold별 결과가 저장됩니다.
DESEQ_OUT_DIR = RUN_DIR / "pydeseq2_cv_results"
DESEQ_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 이미 만들어진 fold별 DESeq 결과를 재사용할지 여부입니다.
USE_EXISTING_DESEQ_RESULTS = True

# 기존 결과가 없을 때 pyDESeq2를 새로 실행할지 여부입니다.
# False이면 기존 결과만 사용합니다.
RUN_PYDESEQ2_IF_MISSING = True

# 기존 결과가 없고 RUN_PYDESEQ2_IF_MISSING=False일 때 동작입니다.
# "error": 오류 발생
# "all_features": 해당 fold에서 전체 feature 사용
DESEQ_ON_MISSING = "error"

# DEG 조건을 만족하는 feature가 0개일 때 동작입니다.
# "error": 오류 발생
# "all_features": 해당 fold에서 전체 feature 사용
DESEQ_EMPTY_SELECTION_ACTION = "error"

# 외부에서 이미 수행된 fold별 DESeq 결과를 사용하고 싶으면 아래에 template을 지정합니다.
# 예: Path("./pydeseq2_cv_results/task1/task1_fold{fold}_deseq.csv")
# 예: Path("./pydeseq2_cv_results/{fold_name}_deseq.csv")  # fold_name = fold_01, fold_02, ...
# None이면 DESEQ_OUT_DIR / "fold_01_deseq.csv" 형식을 사용합니다.
EXTERNAL_DESEQ_RESULT_TEMPLATE = None

DESEQ_N_CPUS = 8
DESEQ_MIN_TOTAL_COUNT_PER_GENE = 10

# 이 노트북의 build_tabnet_rna_classification_dataset()에서 생성되는 표준화된 진단 컬럼입니다.
DESEQ_DIAGNOSIS_COL = "DIAGNOSIS_GROUP"
DESEQ_REFERENCE_LEVEL = "Control"
DESEQ_CONTRAST_CASE = "PD"

# 보정 변수 후보 컬럼입니다.
# 주의: fold별 train split에서 보정 변수를 자동으로 넣으면 sex/clinical_event와 condition이
# 완전히 겹치거나 특정 level 조합이 비어 design matrix가 singular해질 수 있습니다.
# 따라서 기본값은 task1_2_RNA_CV_v2의 목표와 동일하게 condition-only DESeq입니다.
# 필요할 때만 아래 리스트에 "GENDER" 또는 "CLINICAL_EVENT"를 넣으십시오.
DESEQ_SEX_COL = "GENDER"
DESEQ_CLINICAL_EVENT_COL = "CLINICAL_EVENT"
DESEQ_DESIGN_COVARIATES = ["GENDER"]  # 예: ["GENDER"], ["GENDER", "CLINICAL_EVENT"]
DESEQ_RETRY_CONDITION_ONLY_ON_ERROR = True
DESEQ_ON_PYDESEQ2_ERROR = "error"  # "error" 또는 "all_features"

# pyDESeq2 dispersion fitting 안정화를 위한 저발현 gene 필터입니다.
# 각 condition에서 최소 몇 개 샘플 이상 count가 관측되어야 하는지 제한합니다.
# Singular matrix가 나는 경우 2 또는 3으로 올리면 안정성이 증가합니다.
DESEQ_MIN_COUNT_FOR_NONZERO = 1
DESEQ_MIN_SAMPLES_PER_CONDITION = 2

DESEQ_HUDALPHA_COL = "HudAlphaID"
DESEQ_QC_COL = "QCflagIR3"
DESEQ_QC_PASS_VALUE = "pass"

# DEG 선별 기준입니다.
DESEQ_PVAL_COL = "padj"      # 또는 "pvalue"
DESEQ_ALPHA = 0.05
DESEQ_TOP_N = None           # 예: 500, 1000, 2000. None이면 alpha 조건 통과 gene 전체 사용
DESEQ_ABS_LFC_MIN = None     # 예: 0.5, 1.0. None이면 log2FC 기준 미사용
DESEQ_STRIP_VERSION = False  # ENSG000001.1 -> ENSG000001처럼 version 제거가 필요하면 True

# =========================
# General pre-filtering + feature selection 설정
# =========================
# 기본값은 모두 비활성화되어 기존 실행과 동일하게 전체 feature를 사용합니다.
# 사용 흐름:
#   Raw RNA-Seq features
#   -> gene biotype filter(선택)
#   -> fold-wise expression/variance pre-filter(선택)
#   -> fold-wise supervised feature selection(선택)
#
# 권장 실험 예시:
#   GENE_BIOTYPE_FILTER = "protein_coding"
#   PREFILTER_EXPRESSION_ENABLED = True
#   PREFILTER_EXPRESSION_MIN_VALUE = np.log1p(1.0)  # X_df가 log1p(TPM)인 경우 TPM > 1에 해당
#   PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC = 0.10
#   PREFILTER_VARIANCE_TOP_N = 10000
#   FEATURE_SELECTION_METHOD = "deseq"       # 또는 "elasticnet", "ttest", "rf_importance"
#   FEATURE_SELECTION_TOP_N = 500
#
# 주의:
# - annotation 기반 gene biotype filter는 label을 사용하지 않으므로 CV 밖에서 적용해도 됩니다.
# - expression/variance filter와 supervised feature selection은 각 fold의 train split 기준으로만 계산합니다.
# - pretrained TabNet encoder를 사용할 때는 사전학습과 downstream fine-tuning의 feature 목록/순서가 같아야 합니다.

GENE_ANNOTATION_PATH = Path('gencode_v29_gene_annotation_pri.csv')
# 예: Path("./Dataset/RNA/gene_annotation_gencode.csv")
# annotation 파일에는 gene_id, gene_name, gene_biotype 컬럼이 있는 것을 권장합니다.
GENE_ANNOTATION_GENE_ID_COL = "gene_id"
GENE_ANNOTATION_SYMBOL_COL = "gene_name"
GENE_ANNOTATION_BIOTYPE_COL = "gene_biotype"

GENE_BIOTYPE_FILTER = "protein_coding_lncRNA"
# options:
#   "all": 모든 gene feature 사용
#   "protein_coding": protein-coding gene만 사용
#   "protein_coding_lncRNA": protein-coding + lncRNA 계열 gene 사용
#   list/set/tuple: 직접 허용할 gene_biotype 목록 지정
GENE_ANNOTATION_ON_MISSING = "error"  # "error" 또는 "keep_all"
GENE_ID_STRIP_VERSION_FOR_ANNOTATION = True

# fold-wise low-expression pre-filter
PREFILTER_EXPRESSION_ENABLED = True
# X_df가 log1p(TPM)이면 TPM > 1 기준은 np.log1p(1.0)을 사용하세요.
# X_df가 raw TPM이면 1.0을 그대로 사용할 수 있습니다.
PREFILTER_EXPRESSION_MIN_VALUE = np.log1p(1.0)
PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC = 0.10

# fold-wise variance pre-filter
PREFILTER_VARIANCE_TOP_N = None
# options 예시: None, 20000, 10000, 5000, 3000, 1000
# None이면 variance pre-filter를 수행하지 않습니다.

# supervised feature selection
FEATURE_SELECTION_METHOD = "none"
# options:
#   "none": supervised feature selection 미사용
#   "ttest": binary task에서 t-test 기반 top-N gene 선택
#   "anova": ANOVA F-value 기반 top-N gene 선택
#   "deseq": pyDESeq2 결과 기반 top-N 또는 threshold gene 선택
#   "lasso": L1 Logistic Regression coefficient 기반 선택
#   "elasticnet": Elastic Net Logistic Regression coefficient 기반 선택
#   "rf_importance": RandomForest feature importance 기반 선택
#   "xgb_importance": XGBoost feature importance 기반 선택
FEATURE_SELECTION_TOP_N = None
# options 예시:
#   ttest/anova: 200, 500, 1000, 3000
#   deseq: 500, 1000, 3000
#   lasso/elasticnet: 100, 300, 500
#   rf_importance/xgb_importance: 500, 1000
FEATURE_SELECTION_SCOPE = "all"
# options:
#   "none": 어떤 모델에도 적용하지 않음. 기본값
#   "all": 모든 모델에 적용
#   "tabnet": TabNet 계열 모델에만 적용
#   "baselines": baseline 모델에만 적용
#   "custom": FEATURE_SELECTION_MODELS에 명시한 모델에만 적용
FEATURE_SELECTION_MODELS = []  # 예: ["tabnet_scratch", "svm_linear"]

# LASSO / Elastic Net selector 설정
FS_LINEAR_C = 1.0
FS_ELASTICNET_L1_RATIO = 0.5
FS_LINEAR_MAX_ITER = 5000

# RF/XGB importance selector 설정
FS_TREE_N_ESTIMATORS = 300
FS_TREE_MAX_DEPTH = None
FS_TREE_N_JOBS = -1



# Feature selection scope helper는 4번 데이터 구성 셀에서도 필요하므로 설정 셀에서 먼저 정의합니다.
def normalize_model_name_for_feature_selection(model_name):
    if model_name is None:
        return ""
    return str(model_name).lower().strip()


def _model_in_scope_for_feature_selection(model_name, scope, custom_models=None):
    scope = str(scope).lower().strip()
    model = normalize_model_name_for_feature_selection(model_name)

    if scope in ["none", "false", "off", "no"]:
        return False
    if scope in ["all", "true", "on", "yes"]:
        return True
    if scope in ["tabnet", "tabnet_only"]:
        return model.startswith("tabnet")
    if scope in ["baselines", "baseline", "baseline_only"]:
        return not model.startswith("tabnet")
    if scope == "custom":
        custom = {normalize_model_name_for_feature_selection(m) for m in (custom_models or [])}
        return model in custom

    raise ValueError(
        "feature selection scope는 'none', 'all', 'tabnet', 'baselines', 'custom' 중 하나여야 합니다. "
        f"현재 값: {scope!r}"
    )


def should_use_generic_feature_selection_for_model(model_name):
    method = str(FEATURE_SELECTION_METHOD).lower().strip()
    if method in ["none", "false", "off", "no", ""]:
        return False
    return _model_in_scope_for_feature_selection(
        model_name,
        FEATURE_SELECTION_SCOPE,
        FEATURE_SELECTION_MODELS,
    )


def should_use_deseq_for_model(model_name):
    # 새 공통 feature selection 설정에서 method='deseq'를 선택한 경우도 pyDESeq2 사용으로 해석합니다.
    if str(FEATURE_SELECTION_METHOD).lower().strip() == "deseq" and should_use_generic_feature_selection_for_model(model_name):
        return True

    # backward compatibility: 기존 DESeq 전용 설정도 그대로 지원합니다.
    scope = str(DESEQ_FEATURE_SELECTION_SCOPE).lower().strip()
    if bool(USE_DESEQ_FEATURE_SELECTION) and scope == "none":
        scope = "all"
    return _model_in_scope_for_feature_selection(
        model_name,
        scope,
        DESEQ_FEATURE_SELECTION_MODELS,
    )


def should_use_deseq_for_any_model():
    if str(FEATURE_SELECTION_METHOD).lower().strip() == "deseq":
        scope = str(FEATURE_SELECTION_SCOPE).lower().strip()
        if scope not in ["none", "false", "off", "no"]:
            return True

    scope = str(DESEQ_FEATURE_SELECTION_SCOPE).lower().strip()
    if bool(USE_DESEQ_FEATURE_SELECTION) and scope == "none":
        return True
    if scope in ["none", "false", "off", "no"]:
        return False
    if scope in ["all", "true", "on", "yes", "tabnet", "tabnet_only", "baselines", "baseline", "baseline_only"]:
        return True
    if scope == "custom":
        return len(DESEQ_FEATURE_SELECTION_MODELS) > 0
    return True


def prefilter_is_enabled():
    return (
        str(GENE_BIOTYPE_FILTER).lower().strip() not in ["all", "none", "false", "off", "no", ""]
        or bool(PREFILTER_EXPRESSION_ENABLED)
        or PREFILTER_VARIANCE_TOP_N is not None
    )


def get_effective_feature_selection_method(model_name):
    if should_use_generic_feature_selection_for_model(model_name):
        return str(FEATURE_SELECTION_METHOD).lower().strip()

    # 기존 DESeq 전용 설정을 사용하던 실행과 호환합니다.
    if should_use_deseq_for_model(model_name):
        return "deseq"

    return "none"


def get_feature_selection_label(model_name):
    parts = []

    biotype = str(GENE_BIOTYPE_FILTER).lower().strip()
    if biotype not in ["all", "none", "false", "off", "no", ""]:
        parts.append(f"biotype_{biotype}")

    if bool(PREFILTER_EXPRESSION_ENABLED):
        parts.append(f"expr{PREFILTER_EXPRESSION_MIN_VALUE:g}_frac{PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC:g}")

    if PREFILTER_VARIANCE_TOP_N is not None:
        parts.append(f"var_top{int(PREFILTER_VARIANCE_TOP_N)}")

    method = get_effective_feature_selection_method(model_name)
    if method not in ["none", "false", "off", "no", ""]:
        top = f"_top{FEATURE_SELECTION_TOP_N}" if FEATURE_SELECTION_TOP_N is not None else ""
        parts.append(f"{method}{top}")

    return "all_or_pretrained_features" if len(parts) == 0 else "__".join(parts)

print("RUN_DIR:", RUN_DIR.resolve())
print("HAS_STRATIFIED_GROUP_KFOLD:", HAS_STRATIFIED_GROUP_KFOLD)
print("TABNET_TRAIN_MODE:", TABNET_TRAIN_MODE)
print("PRETRAINED_TABNET_RUN_DIR:", PRETRAINED_TABNET_RUN_DIR)
print("FREEZE_PRETRAINED_ENCODER:", FREEZE_PRETRAINED_ENCODER)
print("FREEZE_PRETRAINED_EMBEDDER:", FREEZE_PRETRAINED_EMBEDDER)
print("DESEQ_FEATURE_SELECTION_SCOPE:", DESEQ_FEATURE_SELECTION_SCOPE)
print("DESEQ_FEATURE_SELECTION_MODELS:", DESEQ_FEATURE_SELECTION_MODELS)
print("DESEQ_OUT_DIR:", DESEQ_OUT_DIR.resolve())
print("GENE_BIOTYPE_FILTER:", GENE_BIOTYPE_FILTER)
print("PREFILTER_EXPRESSION_ENABLED:", PREFILTER_EXPRESSION_ENABLED)
print("PREFILTER_VARIANCE_TOP_N:", PREFILTER_VARIANCE_TOP_N)
print("FEATURE_SELECTION_METHOD:", FEATURE_SELECTION_METHOD)
print("FEATURE_SELECTION_SCOPE:", FEATURE_SELECTION_SCOPE)
print("FEATURE_SELECTION_TOP_N:", FEATURE_SELECTION_TOP_N)


## 4. 교차검증 입력 데이터 구성

앞에서 만든 `tabnet_data`에서 전체 feature matrix, label, 환자 ID group을 꺼냅니다.  
교차검증은 동일 환자가 train/valid fold에 동시에 들어가지 않도록 `groups`를 사용합니다.

DESeq feature selection을 fold마다 다르게 적용할 수 있도록, `X_all` numpy array뿐 아니라 `X_all_df` DataFrame도 함께 유지합니다.


In [ ]:
# 4. 교차검증용 전체 데이터 구성

# X_all_df는 log1p/fillna까지 적용된 DataFrame입니다.
# fold별 DEG feature selection을 적용하려면 column 이름이 필요하므로 DataFrame 형태를 유지합니다.
X_all_df = tabnet_data["X_all_df"].reset_index(drop=True)
X_all = X_all_df.to_numpy(dtype=np.float32)

y_all = tabnet_data["y_all"].astype(np.int64)
groups_all = tabnet_data["groups"]
filtered_df = tabnet_data["filtered_df"].reset_index(drop=True)

# pyDESeq2 feature selection에서 사용할 sample key가 filtered_df까지 유지되었는지 확인합니다.
if should_use_deseq_for_any_model():
    if DESEQ_HUDALPHA_COL not in filtered_df.columns:
        raise ValueError(
            f"filtered_df에 {DESEQ_HUDALPHA_COL!r} 컬럼이 없습니다. "
            "meta_df -> meta_sub -> merged_df -> filtered_df로 HudAlphaID가 유지되는지 확인하세요."
        )
    print(f"{DESEQ_HUDALPHA_COL} in filtered_df: True")
    print(f"{DESEQ_HUDALPHA_COL} unique:", filtered_df[DESEQ_HUDALPHA_COL].nunique(dropna=True))
    print(f"{DESEQ_HUDALPHA_COL} 예시:", filtered_df[DESEQ_HUDALPHA_COL].dropna().astype(str).head().tolist())

feature_cols_tabnet = tabnet_data["feature_cols"]
label_map = tabnet_data["label_map"]
inverse_label_map = tabnet_data["inverse_label_map"]
patient_col = tabnet_data["patient_col"]

class_labels = sorted(inverse_label_map.keys())
class_names = [inverse_label_map[i] for i in class_labels]

print("X_all_df:", X_all_df.shape)
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print("groups_all:", groups_all.shape)
print("class_names:", class_names)
print("patients:", len(np.unique(groups_all)))

dataset_summary = {
    "n_samples": int(X_all.shape[0]),
    "n_features_total": int(X_all.shape[1]),
    "n_patients": int(len(np.unique(groups_all))),
    "class_distribution": pd.Series(y_all).map(inverse_label_map).value_counts().to_dict(),
    "label_map": label_map,
    "patient_col": patient_col,
    "n_splits": int(N_SPLITS),
    "scale_tabnet_within_fold": bool(SCALE_TABNET_WITHIN_FOLD),
    "baseline_models": BASELINE_MODELS,
    "deseq_feature_selection_scope": DESEQ_FEATURE_SELECTION_SCOPE,
    "deseq_feature_selection_models": DESEQ_FEATURE_SELECTION_MODELS,
    "deseq_reference_level": DESEQ_REFERENCE_LEVEL,
    "deseq_contrast_case": DESEQ_CONTRAST_CASE,
    "deseq_pval_col": DESEQ_PVAL_COL,
    "deseq_alpha": DESEQ_ALPHA,
    "deseq_top_n": DESEQ_TOP_N,
    "deseq_abs_lfc_min": DESEQ_ABS_LFC_MIN,
}

with open(RUN_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(dataset_summary, f, indent=2, ensure_ascii=False)

pd.DataFrame({"feature": feature_cols_tabnet}).to_csv(RUN_DIR / "feature_columns_all.csv", index=False)
pd.DataFrame({"label": class_labels, "class_name": class_names}).to_csv(RUN_DIR / "label_map.csv", index=False)

display(pd.Series(y_all).map(inverse_label_map).value_counts().rename("count").to_frame())


## 5. 평가 및 저장 유틸리티

아래 함수들은 fold별 결과를 일관된 구조로 저장합니다.

저장되는 주요 파일은 다음과 같습니다.

- `metrics.csv`: accuracy, balanced accuracy, macro/weighted F1 등
- `classification_report.csv`
- `confusion_matrix.csv`, `confusion_matrix.png`
- `predictions.csv`
- `feature_importance.csv`
- `history.csv`, `training_curves.png` 또는 metric별 curve
- 모델 파일


In [ ]:
# 5. 평가 및 저장 유틸리티

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def get_cv_splitter(n_splits, random_state=42, use_groups=True):
    """
    환자 단위 누수를 막기 위해 기본적으로 StratifiedGroupKFold를 사용합니다.
    설치된 scikit-learn 버전에서 StratifiedGroupKFold가 없으면 GroupKFold로 fallback합니다.
    """
    if use_groups:
        if HAS_STRATIFIED_GROUP_KFOLD:
            return StratifiedGroupKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=random_state,
            )
        warnings.warn(
            "StratifiedGroupKFold를 사용할 수 없어 GroupKFold로 대체합니다. "
            "class 비율이 fold별로 완전히 보존되지 않을 수 있습니다."
        )
        return GroupKFold(n_splits=n_splits)

    return StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )


def compute_metric_dict(y_true, y_pred, y_score=None, model_name=None, fold=None):
    metric = {
        "model": model_name,
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }

    for avg in ["macro", "weighted"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=avg,
            zero_division=0,
        )
        metric[f"precision_{avg}"] = precision
        metric[f"recall_{avg}"] = recall
        metric[f"f1_{avg}"] = f1

    # binary / multiclass ROC-AUC를 계산할 수 있는 경우만 저장합니다.
    if y_score is not None:
        try:
            if len(np.unique(y_true)) == 2:
                if y_score.ndim == 2 and y_score.shape[1] >= 2:
                    metric["roc_auc"] = roc_auc_score(y_true, y_score[:, 1])
                else:
                    metric["roc_auc"] = roc_auc_score(y_true, y_score)
            elif y_score.ndim == 2 and y_score.shape[1] == len(class_labels):
                metric["roc_auc_ovr_macro"] = roc_auc_score(
                    y_true,
                    y_score,
                    multi_class="ovr",
                    average="macro",
                )
        except Exception as e:
            metric["roc_auc_error"] = str(e)

    return metric


def make_prediction_df(eval_indices, y_true, y_pred, y_score=None):
    meta_cols = [
        c for c in [
            "SAMPLE_ID_norm",
            "PATNO_norm",
            "DIAGNOSIS_GROUP",
            "DIAGNOSIS",
            "CLINICAL_EVENT",
            "VISIT",
            "VISIT_norm",
            "QCflagIR3",
        ]
        if c in filtered_df.columns
    ]

    pred_df = filtered_df.iloc[eval_indices][meta_cols].reset_index(drop=True).copy()
    pred_df["y_true"] = y_true
    pred_df["y_true_name"] = [inverse_label_map[int(v)] for v in y_true]
    pred_df["y_pred"] = y_pred
    pred_df["y_pred_name"] = [inverse_label_map[int(v)] for v in y_pred]
    pred_df["correct"] = pred_df["y_true"] == pred_df["y_pred"]

    if y_score is not None:
        y_score = np.asarray(y_score)
        if y_score.ndim == 2:
            for j in range(y_score.shape[1]):
                col_name = f"score_or_prob_class_{j}_{inverse_label_map.get(j, j)}"
                pred_df[col_name] = y_score[:, j]
        else:
            pred_df["score_or_prob"] = y_score

    return pred_df


def save_evaluation_outputs(
    model_name,
    fold,
    fold_dir,
    y_true,
    y_pred,
    eval_indices,
    y_score=None,
):
    fold_dir = Path(fold_dir)
    fold_dir.mkdir(parents=True, exist_ok=True)

    metric = compute_metric_dict(
        y_true=y_true,
        y_pred=y_pred,
        y_score=y_score,
        model_name=model_name,
        fold=fold,
    )
    pd.DataFrame([metric]).to_csv(fold_dir / "metrics.csv", index=False)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).T
    report_df.to_csv(fold_dir / "classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(fold_dir / "confusion_matrix.csv")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(values_format="d", ax=ax)
    ax.set_title(f"{model_name} Fold {fold} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(fold_dir / "confusion_matrix.png", dpi=200)
    plt.close(fig)

    pred_df = make_prediction_df(eval_indices, y_true, y_pred, y_score=y_score)
    pred_df.to_csv(fold_dir / "predictions.csv", index=False)

    save_json(metric, fold_dir / "metrics.json")

    return metric


def save_feature_importance(importance, feature_names, out_path, top_n=None):
    if importance is None:
        return None

    importance = np.asarray(importance).reshape(-1)
    if len(importance) != len(feature_names):
        print(f"feature importance 길이가 feature 개수와 다릅니다: {len(importance)} vs {len(feature_names)}")
        return None

    fi_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance,
    }).sort_values("importance", ascending=False)

    fi_df.to_csv(out_path, index=False)

    if top_n is not None:
        display(fi_df.head(top_n))

    return fi_df


def plot_and_save_top_importance(fi_df, out_path, title, top_n=30):
    if fi_df is None or len(fi_df) == 0:
        return

    top_df = fi_df.head(top_n).iloc[::-1]

    fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.25)))
    ax.barh(top_df["feature"], top_df["importance"])
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def save_tabnet_history(history, fold_dir):
    history_df = pd.DataFrame(history)
    history_df.to_csv(fold_dir / "history.csv", index=False)

    if len(history_df) == 0:
        return history_df

    if "loss" in history_df.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(history_df["loss"], label="train loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Training Loss")
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(fold_dir / "training_loss.png", dpi=200)
        plt.close(fig)

    metric_cols = [
        col for col in history_df.columns
        if ("accuracy" in col or "balanced_accuracy" in col or "auc" in col)
    ]

    if metric_cols:
        fig, ax = plt.subplots(figsize=(8, 5))
        for col in metric_cols:
            ax.plot(history_df[col], label=col)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Metric")
        ax.set_title("Validation Metrics")
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(fold_dir / "training_metrics.png", dpi=200)
        plt.close(fig)

    return history_df


def summarize_metrics(metrics_df, group_cols=("model",)):
    numeric_cols = metrics_df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != "fold"]

    if not numeric_cols:
        return pd.DataFrame()

    summary = (
        metrics_df
        .groupby(list(group_cols))[numeric_cols]
        .agg(["mean", "std", "min", "max"])
    )
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    return summary.reset_index()


## 5-1. fold-wise pyDESeq2 DEG feature selection 유틸리티

아래 함수들은 각 CV fold에서 **train split만 사용해 pyDESeq2 차등발현분석을 수행**하고, 해당 fold에서 사용할 DEG feature column을 선택합니다.

- 모델 입력은 기존과 동일하게 `log1p(TPM)` feature를 사용합니다.
- pyDESeq2에는 `featureCounts` 기반 raw count matrix를 사용합니다.
- 기존 fold별 DESeq 결과 CSV가 있으면 재사용할 수 있습니다.
- TabNet과 baseline 모두 같은 fold의 같은 DEG feature set을 사용합니다.


In [ ]:
# 5-1. pyDESeq2 기반 fold-wise DEG feature selection 유틸리티

try:
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    from pydeseq2.default_inference import DefaultInference
    PYDESEQ2_AVAILABLE = True
    PYDESEQ2_IMPORT_ERROR = None
except Exception as e:
    PYDESEQ2_AVAILABLE = False
    PYDESEQ2_IMPORT_ERROR = e


def ensure_pydeseq2_available():
    if not PYDESEQ2_AVAILABLE:
        raise ImportError(
            "pyDESeq2를 import하지 못했습니다. USE_DESEQ_FEATURE_SELECTION=True로 사용하려면 "
            "pydeseq2를 설치하거나, USE_DESEQ_FEATURE_SELECTION=False로 설정하세요. "
            f"원래 오류: {PYDESEQ2_IMPORT_ERROR}"
        )


def extract_hudalphaid(text):
    m = re.search(r"5104-SL-\d+", str(text))
    if m is None:
        raise ValueError(f"HudAlphaID를 찾을 수 없습니다: {text}")
    return m.group(0)


def normalize_hudalphaid_for_match(x):
    """
    metadata/count matrix 양쪽의 sample ID를 비교 가능한 문자열로 정리합니다.
    5104-SL-... 패턴이 있으면 해당 부분만 추출하고, 없으면 strip한 문자열을 사용합니다.
    """
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    m = re.search(r"5104-SL-\d+", x)
    if m is not None:
        return m.group(0)
    return x


def read_featurecounts_txt(path, strip_gene_version=False):
    df = pd.read_csv(path, sep="\t", comment="#", low_memory=False)

    if "Geneid" not in df.columns:
        raise ValueError(f"'Geneid' 열이 없습니다: {path}")

    count_col = df.columns[-1]
    sample_id = extract_hudalphaid(Path(path).name)

    out = df[["Geneid", count_col]].copy()
    out.columns = ["gene_id", sample_id]

    if strip_gene_version:
        out["gene_id"] = out["gene_id"].str.replace(r"\.\d+$", "", regex=True)
        out = out.groupby("gene_id", as_index=False)[sample_id].sum()

    return out.set_index("gene_id")


def build_counts_matrix(counts_dir, strip_gene_version=False):
    txt_files = sorted(Path(counts_dir).glob("*.txt"))

    if len(txt_files) == 0:
        raise FileNotFoundError(f"featureCounts txt 파일이 없습니다: {counts_dir}")

    count_tables = []
    for fp in txt_files:
        count_tables.append(
            read_featurecounts_txt(fp, strip_gene_version=strip_gene_version)
        )

    counts_gene_by_sample = pd.concat(count_tables, axis=1, join="outer").fillna(0)
    counts_df = counts_gene_by_sample.T
    counts_df = counts_df.round().astype(int)
    counts_df.index.name = "HudAlphaID"

    return counts_df


COUNTS_DF_CACHE = None


def get_counts_df():
    global COUNTS_DF_CACHE

    if COUNTS_DF_CACHE is None:
        COUNTS_DF_CACHE = build_counts_matrix(
            COUNTS_DIR,
            strip_gene_version=DESEQ_STRIP_VERSION,
        )

    return COUNTS_DF_CACHE.copy()


def normalize_feature_id_for_match(x, strip_version=False):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if strip_version:
        x = re.sub(r"\.\d+$", "", x)

    return x


def build_rank_check_matrix(design_meta, design_cols):
    """
    pyDESeq2 design_factors에 넣기 전에 대략적인 full-rank 여부를 확인하기 위한
    dummy-coded design matrix를 만듭니다.
    """
    X_design = pd.get_dummies(design_meta[design_cols], drop_first=True, dtype=float)
    X_design.insert(0, "intercept", 1.0)
    return X_design


def is_full_rank_design(design_meta, design_cols):
    if len(design_meta) == 0:
        return False
    X_design = build_rank_check_matrix(design_meta, design_cols)
    rank = np.linalg.matrix_rank(X_design.to_numpy(dtype=float))
    return rank == X_design.shape[1]


def build_deseq_design(train_meta):
    """
    DESeq design을 구성합니다.
    기본값은 condition-only입니다. DESEQ_DESIGN_COVARIATES에 지정한 보정 변수만
    후보로 넣고, full-rank가 깨지는 covariate는 자동 제외합니다.
    """
    design_cols = ["condition"]

    covariate_specs = []
    requested = set([str(x).upper() for x in DESEQ_DESIGN_COVARIATES])

    if DESEQ_SEX_COL.upper() in requested or "SEX" in requested or "GENDER" in requested:
        covariate_specs.append((DESEQ_SEX_COL, "sex"))

    if DESEQ_CLINICAL_EVENT_COL.upper() in requested or "CLINICAL_EVENT" in requested:
        covariate_specs.append((DESEQ_CLINICAL_EVENT_COL, "clinical_event"))

    for source_col, design_col in covariate_specs:
        if source_col not in train_meta.columns:
            print(f"[DESeq] covariate skip: {source_col} 컬럼이 없습니다.")
            continue

        if train_meta[source_col].dropna().nunique() < 2:
            print(f"[DESeq] covariate skip: {source_col} 값이 2개 미만입니다.")
            continue

        train_meta[design_col] = pd.Categorical(train_meta[source_col])
        candidate_cols = design_cols[:-1] + [design_col] + ["condition"]
        candidate_meta = train_meta[candidate_cols].dropna().copy()

        if candidate_meta["condition"].dropna().nunique() < 2:
            print(f"[DESeq] covariate skip: {source_col} 추가 후 condition level이 부족합니다.")
            continue

        if is_full_rank_design(candidate_meta, candidate_cols):
            design_cols = candidate_cols
        else:
            print(f"[DESeq] covariate skip: {source_col} 추가 시 design matrix가 full-rank가 아닙니다.")

    return design_cols


def apply_condition_wise_gene_filter(counts_sub, design_meta):
    """
    pyDESeq2에서 gene-wise dispersion 추정 중 singular matrix가 나는 것을 줄이기 위해
    각 condition에서 최소 관측 샘플 수를 만족하는 gene만 남깁니다.
    """
    min_samples = int(DESEQ_MIN_SAMPLES_PER_CONDITION)
    min_count = int(DESEQ_MIN_COUNT_FOR_NONZERO)

    if min_samples <= 0:
        return counts_sub

    keep = pd.Series(True, index=counts_sub.columns)
    for level in [DESEQ_REFERENCE_LEVEL, DESEQ_CONTRAST_CASE]:
        idx = design_meta.index[design_meta["condition"] == level]
        if len(idx) == 0:
            return counts_sub.iloc[:, 0:0]
        group_keep = (counts_sub.loc[idx] >= min_count).sum(axis=0) >= min_samples
        keep &= group_keep

    before = counts_sub.shape[1]
    counts_sub = counts_sub.loc[:, keep].copy()
    after = counts_sub.shape[1]
    print(f"[DESeq] condition-wise gene filter: {before} -> {after} genes "
          f"(min_count={min_count}, min_samples_per_condition={min_samples})")
    return counts_sub


def get_deseq_result_path(fold):
    fold_name = f"fold_{fold:02d}"

    if EXTERNAL_DESEQ_RESULT_TEMPLATE is not None:
        template = str(EXTERNAL_DESEQ_RESULT_TEMPLATE)
        return Path(template.format(fold=fold, fold_name=fold_name))

    return DESEQ_OUT_DIR / f"{fold_name}_deseq.csv"


def run_or_load_fold_deseq(
    fold_train_df,
    fold,
    use_existing=True,
    run_if_missing=True,
    counts_df=None,
):
    out_csv = get_deseq_result_path(fold)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    if use_existing and out_csv.exists():
        print("기존 DESeq 결과 사용:", out_csv)
        return pd.read_csv(out_csv)

    if not run_if_missing:
        msg = f"기존 DESeq 결과가 없습니다: {out_csv}"

        if DESEQ_ON_MISSING == "all_features":
            print("[DESeq] " + msg + " -> 전체 feature 사용")
            return None

        raise FileNotFoundError(msg)

    ensure_pydeseq2_available()

    if counts_df is None:
        counts_df = get_counts_df()

    train_meta = fold_train_df.copy()

    required_cols = [DESEQ_HUDALPHA_COL, DESEQ_DIAGNOSIS_COL]
    missing_cols = [c for c in required_cols if c not in train_meta.columns]
    if missing_cols:
        raise ValueError(f"DESeq에 필요한 컬럼이 filtered_df에 없습니다: {missing_cols}")

    train_meta = train_meta.drop_duplicates(subset=[DESEQ_HUDALPHA_COL]).copy()
    train_meta[DESEQ_HUDALPHA_COL] = train_meta[DESEQ_HUDALPHA_COL].apply(normalize_hudalphaid_for_match)
    train_meta = train_meta.dropna(subset=[DESEQ_HUDALPHA_COL]).copy()
    train_meta[DESEQ_HUDALPHA_COL] = train_meta[DESEQ_HUDALPHA_COL].astype(str)

    counts_df = counts_df.copy()
    counts_df.index = pd.Index(
        [normalize_hudalphaid_for_match(x) for x in counts_df.index],
        name=DESEQ_HUDALPHA_COL,
    ).astype(str)

    common_samples = counts_df.index.intersection(train_meta[DESEQ_HUDALPHA_COL])

    if len(common_samples) == 0:
        counts_examples = counts_df.index[:10].tolist()
        meta_examples = train_meta[DESEQ_HUDALPHA_COL].dropna().astype(str).head(10).tolist()
        raise ValueError(
            f"[DESeq] fold {fold}: count matrix와 metadata의 공통 HudAlphaID가 없습니다.\n"
            f"counts_df index 예시: {counts_examples}\n"
            f"metadata {DESEQ_HUDALPHA_COL} 예시: {meta_examples}\n"
            "meta_df -> meta_sub -> merged_df -> filtered_df에 보존된 HudAlphaID가 "
            "featureCounts 파일명에서 추출한 5104-SL-... ID와 같은지 확인하세요."
        )

    train_meta = train_meta.set_index(DESEQ_HUDALPHA_COL, drop=True).loc[common_samples].copy()
    counts_sub = counts_df.loc[common_samples].copy()

    # QC 필터
    if DESEQ_QC_COL in train_meta.columns:
        train_meta = train_meta[train_meta[DESEQ_QC_COL] == DESEQ_QC_PASS_VALUE].copy()
        counts_sub = counts_sub.loc[train_meta.index].copy()

    # condition 필터
    train_meta = train_meta[
        train_meta[DESEQ_DIAGNOSIS_COL].isin([DESEQ_REFERENCE_LEVEL, DESEQ_CONTRAST_CASE])
    ].copy()
    counts_sub = counts_sub.loc[train_meta.index].copy()

    if len(train_meta) == 0:
        raise ValueError(f"[DESeq] fold {fold}: condition 필터 후 남은 샘플이 없습니다.")

    # zero count sample 제거
    sample_keep = counts_sub.sum(axis=1) > 0
    counts_sub = counts_sub.loc[sample_keep].copy()
    train_meta = train_meta.loc[sample_keep].copy()

    # low-count gene 제거
    genes_to_keep = counts_sub.columns[
        counts_sub.sum(axis=0) >= DESEQ_MIN_TOTAL_COUNT_PER_GENE
    ]
    counts_sub = counts_sub[genes_to_keep].copy()

    if counts_sub.shape[1] == 0:
        raise ValueError(f"[DESeq] fold {fold}: low-count filtering 후 남은 gene이 없습니다.")

    # design 구성
    train_meta["condition"] = pd.Categorical(
        train_meta[DESEQ_DIAGNOSIS_COL],
        categories=[DESEQ_REFERENCE_LEVEL, DESEQ_CONTRAST_CASE],
    )

    if train_meta["condition"].dropna().nunique() < 2:
        raise ValueError(f"[DESeq] fold {fold}: condition이 하나만 남았습니다.")

    # design 구성: 기본은 condition-only입니다.
    # 보정 변수는 DESEQ_DESIGN_COVARIATES에 명시된 경우에만 후보로 사용하며,
    # full-rank가 깨지는 covariate는 자동으로 제외합니다.
    design_cols = build_deseq_design(train_meta)

    design_meta = train_meta[design_cols].copy()
    valid = ~design_meta.isna().any(axis=1)
    design_meta = design_meta.loc[valid].copy()
    counts_sub = counts_sub.loc[valid].copy()

    if design_meta["condition"].dropna().nunique() < 2:
        raise ValueError(f"[DESeq] fold {fold}: 유효 샘플 필터 후 condition이 하나만 남았습니다.")

    if not is_full_rank_design(design_meta, design_cols):
        print("[DESeq] design matrix가 full-rank가 아니므로 condition-only로 재설정합니다.")
        design_cols = ["condition"]
        design_meta = train_meta[design_cols].copy()
        valid = ~design_meta.isna().any(axis=1)
        design_meta = design_meta.loc[valid].copy()
        counts_sub = counts_sub.loc[valid].copy()

    # gene-wise dispersion fitting 안정화를 위한 condition-wise gene filter
    counts_sub = apply_condition_wise_gene_filter(counts_sub, design_meta)
    if counts_sub.shape[1] == 0:
        msg = f"[DESeq] fold {fold}: condition-wise filtering 후 남은 gene이 없습니다."
        if DESEQ_ON_PYDESEQ2_ERROR == "all_features":
            print(msg + " -> 전체 feature 사용")
            return None
        raise ValueError(msg)

    print("=" * 80)
    print(f"[DESeq] fold {fold}")
    print("samples:", counts_sub.shape[0])
    print("genes:", counts_sub.shape[1])
    print("condition:")
    print(design_meta["condition"].value_counts(dropna=False))
    print("design_cols:", design_cols)
    print("=" * 80)

    inference = DefaultInference(n_cpus=DESEQ_N_CPUS)

    def _fit_deseq_once(_design_cols, _design_meta, _counts_sub):
        _dds = DeseqDataSet(
            counts=_counts_sub,
            metadata=_design_meta,
            design_factors=_design_cols,
            ref_level=["condition", DESEQ_REFERENCE_LEVEL],
            refit_cooks=True,
            inference=inference,
        )
        _dds.deseq2()
        return _dds

    try:
        dds = _fit_deseq_once(design_cols, design_meta, counts_sub)
    except np.linalg.LinAlgError as e:
        print(f"[DESeq] LinAlgError during dds.deseq2(): {e}")
        if DESEQ_RETRY_CONDITION_ONLY_ON_ERROR and design_cols != ["condition"]:
            print("[DESeq] condition-only design으로 재시도합니다.")
            design_cols = ["condition"]
            design_meta = train_meta[design_cols].copy()
            valid = ~design_meta.isna().any(axis=1)
            design_meta = design_meta.loc[valid].copy()
            counts_sub = counts_sub.loc[valid].copy()
            counts_sub = apply_condition_wise_gene_filter(counts_sub, design_meta)
            dds = _fit_deseq_once(design_cols, design_meta, counts_sub)
        elif DESEQ_ON_PYDESEQ2_ERROR == "all_features":
            print("[DESeq] pyDESeq2 실패 -> 전체 feature 사용")
            return None
        else:
            raise

    stat_res = DeseqStats(
        dds,
        contrast=["condition", DESEQ_CONTRAST_CASE, DESEQ_REFERENCE_LEVEL],
        inference=inference,
    )
    stat_res.summary()

    res = stat_res.results_df.copy()
    res.index.name = "gene_id"
    res = res.reset_index().sort_values(DESEQ_PVAL_COL, na_position="last")

    res.to_csv(out_csv, index=False)
    print("DESeq 결과 저장:", out_csv)

    return res


def get_deseq_selected_feature_cols(
    feature_cols,
    deseq_df,
    id_col="gene_id",
    pval_col="padj",
    alpha=0.05,
    top_n=None,
    abs_lfc_min=None,
    strip_version=False,
    verbose=True,
):
    if deseq_df is None:
        return feature_cols, None

    work = deseq_df.copy()

    if id_col not in work.columns:
        raise ValueError(f"DESeq 결과에 {id_col!r} 컬럼이 없습니다.")

    if pval_col not in work.columns:
        raise ValueError(f"DESeq 결과에 {pval_col!r} 컬럼이 없습니다.")

    work = work.dropna(subset=[pval_col]).copy()
    work["_match_id"] = work[id_col].apply(
        lambda x: normalize_feature_id_for_match(x, strip_version=strip_version)
    )

    if abs_lfc_min is not None:
        if "log2FoldChange" not in work.columns:
            raise ValueError("abs_lfc_min을 사용하려면 log2FoldChange 컬럼이 필요합니다.")
        work = work[work["log2FoldChange"].abs() >= abs_lfc_min].copy()

    if alpha is not None:
        work = work[work[pval_col] <= alpha].copy()

    work = work.sort_values(pval_col, ascending=True).copy()

    if top_n is not None:
        work = work.head(top_n).copy()

    feature_map = {}
    for col in feature_cols:
        key = normalize_feature_id_for_match(col, strip_version=strip_version)
        if key not in feature_map:
            feature_map[key] = col

    selected_feature_cols = []
    matched_ids = []

    for gid in work["_match_id"].tolist():
        if gid in feature_map:
            selected_feature_cols.append(feature_map[gid])
            matched_ids.append(gid)

    matched_deseq_df = work[work["_match_id"].isin(matched_ids)].copy()

    if verbose:
        print("DESeq prefilter summary")
        print("원래 feature 수:", len(feature_cols))
        print("DESeq 조건 통과 gene 수:", len(work))
        print("TPM feature와 매칭된 gene 수:", len(selected_feature_cols))

    if len(selected_feature_cols) == 0:
        msg = (
            "DESeq 조건을 만족하면서 TPM feature와 매칭되는 gene이 없습니다. "
            "DESEQ_ALPHA, DESEQ_TOP_N, DESEQ_ABS_LFC_MIN, DESEQ_STRIP_VERSION 설정을 확인하세요."
        )
        if DESEQ_EMPTY_SELECTION_ACTION == "all_features":
            print("[DESeq] " + msg + " -> 전체 feature 사용")
            return feature_cols, matched_deseq_df
        raise ValueError(msg)

    return selected_feature_cols, matched_deseq_df




def normalize_model_name_for_feature_selection(model_name):
    if model_name is None:
        return ""
    return str(model_name).lower().strip()


def _model_in_scope_for_feature_selection(model_name, scope, custom_models=None):
    scope = str(scope).lower().strip()
    model = normalize_model_name_for_feature_selection(model_name)

    if scope in ["none", "false", "off", "no"]:
        return False
    if scope in ["all", "true", "on", "yes"]:
        return True
    if scope in ["tabnet", "tabnet_only"]:
        return model.startswith("tabnet")
    if scope in ["baselines", "baseline", "baseline_only"]:
        return not model.startswith("tabnet")
    if scope == "custom":
        custom = {normalize_model_name_for_feature_selection(m) for m in (custom_models or [])}
        return model in custom

    raise ValueError(
        "feature selection scope는 'none', 'all', 'tabnet', 'baselines', 'custom' 중 하나여야 합니다. "
        f"현재 값: {scope!r}"
    )


def should_use_generic_feature_selection_for_model(model_name):
    method = str(FEATURE_SELECTION_METHOD).lower().strip()
    if method in ["none", "false", "off", "no", ""]:
        return False
    return _model_in_scope_for_feature_selection(
        model_name,
        FEATURE_SELECTION_SCOPE,
        FEATURE_SELECTION_MODELS,
    )


def should_use_deseq_for_model(model_name):
    if str(FEATURE_SELECTION_METHOD).lower().strip() == "deseq" and should_use_generic_feature_selection_for_model(model_name):
        return True

    scope = str(DESEQ_FEATURE_SELECTION_SCOPE).lower().strip()
    if bool(USE_DESEQ_FEATURE_SELECTION) and scope == "none":
        scope = "all"
    return _model_in_scope_for_feature_selection(
        model_name,
        scope,
        DESEQ_FEATURE_SELECTION_MODELS,
    )


def should_use_deseq_for_any_model():
    if str(FEATURE_SELECTION_METHOD).lower().strip() == "deseq":
        scope = str(FEATURE_SELECTION_SCOPE).lower().strip()
        if scope not in ["none", "false", "off", "no"]:
            return True

    scope = str(DESEQ_FEATURE_SELECTION_SCOPE).lower().strip()
    if bool(USE_DESEQ_FEATURE_SELECTION) and scope == "none":
        return True
    if scope in ["none", "false", "off", "no"]:
        return False
    if scope in ["all", "true", "on", "yes", "tabnet", "tabnet_only", "baselines", "baseline", "baseline_only"]:
        return True
    if scope == "custom":
        return len(DESEQ_FEATURE_SELECTION_MODELS) > 0
    return True


def prefilter_is_enabled():
    return (
        str(GENE_BIOTYPE_FILTER).lower().strip() not in ["all", "none", "false", "off", "no", ""]
        or bool(PREFILTER_EXPRESSION_ENABLED)
        or PREFILTER_VARIANCE_TOP_N is not None
    )


def get_effective_feature_selection_method(model_name):
    if should_use_generic_feature_selection_for_model(model_name):
        return str(FEATURE_SELECTION_METHOD).lower().strip()
    if should_use_deseq_for_model(model_name):
        return "deseq"
    return "none"


def get_feature_selection_label(model_name):
    parts = []

    biotype = str(GENE_BIOTYPE_FILTER).lower().strip()
    if biotype not in ["all", "none", "false", "off", "no", ""]:
        parts.append(f"biotype_{biotype}")

    if bool(PREFILTER_EXPRESSION_ENABLED):
        parts.append(f"expr{PREFILTER_EXPRESSION_MIN_VALUE:g}_frac{PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC:g}")

    if PREFILTER_VARIANCE_TOP_N is not None:
        parts.append(f"var_top{int(PREFILTER_VARIANCE_TOP_N)}")

    method = get_effective_feature_selection_method(model_name)
    if method not in ["none", "false", "off", "no", ""]:
        top = f"_top{FEATURE_SELECTION_TOP_N}" if FEATURE_SELECTION_TOP_N is not None else ""
        parts.append(f"{method}{top}")

    return "all_or_pretrained_features" if len(parts) == 0 else "__".join(parts)


def _strip_gene_version(x):
    x = str(x)
    return x.split(".")[0] if "." in x else x


def _load_gene_annotation():
    if GENE_ANNOTATION_PATH is None:
        return None

    path = Path(GENE_ANNOTATION_PATH)
    if not path.exists():
        action = str(GENE_ANNOTATION_ON_MISSING).lower().strip()
        if action == "keep_all":
            print(f"경고: gene annotation 파일을 찾지 못해 biotype filter를 건너뜁니다: {path}")
            return None
        raise FileNotFoundError(f"GENE_ANNOTATION_PATH를 찾지 못했습니다: {path}")

    if path.suffix.lower() in [".tsv", ".txt"]:
        ann = pd.read_csv(path, sep="\t")
    else:
        ann = pd.read_csv(path)

    required = [GENE_ANNOTATION_BIOTYPE_COL]
    for col in required:
        if col not in ann.columns:
            raise ValueError(f"annotation 파일에 필요한 컬럼이 없습니다: {col}")

    return ann


def _allowed_biotypes():
    filt = GENE_BIOTYPE_FILTER
    if isinstance(filt, (list, tuple, set)):
        return {str(x) for x in filt}

    filt = str(filt).lower().strip()
    if filt in ["all", "none", "false", "off", "no", ""]:
        return None
    if filt == "protein_coding":
        return {"protein_coding"}
    if filt in ["protein_coding_lncrna", "protein_coding+lncrna", "protein_coding_lncRNA".lower()]:
        return {
            "protein_coding",
            "lncRNA",
            "lincRNA",
            "antisense",
            "processed_transcript",
            "sense_intronic",
            "sense_overlapping",
            "3prime_overlapping_ncRNA",
            "non_coding",
        }
    return {GENE_BIOTYPE_FILTER}


def apply_gene_biotype_filter(feature_cols):
    allowed = _allowed_biotypes()
    if allowed is None:
        return list(feature_cols), pd.DataFrame(), {
            "step": "gene_biotype_filter",
            "enabled": False,
            "n_before": int(len(feature_cols)),
            "n_after": int(len(feature_cols)),
        }

    ann = _load_gene_annotation()
    if ann is None:
        return list(feature_cols), pd.DataFrame(), {
            "step": "gene_biotype_filter",
            "enabled": False,
            "reason": "annotation_missing_keep_all",
            "n_before": int(len(feature_cols)),
            "n_after": int(len(feature_cols)),
        }

    key_to_biotype = {}

    if GENE_ANNOTATION_GENE_ID_COL in ann.columns:
        for gene_id, biotype in zip(ann[GENE_ANNOTATION_GENE_ID_COL], ann[GENE_ANNOTATION_BIOTYPE_COL]):
            key_to_biotype[str(gene_id)] = str(biotype)
            if GENE_ID_STRIP_VERSION_FOR_ANNOTATION:
                key_to_biotype[_strip_gene_version(gene_id)] = str(biotype)

    if GENE_ANNOTATION_SYMBOL_COL in ann.columns:
        for symbol, biotype in zip(ann[GENE_ANNOTATION_SYMBOL_COL], ann[GENE_ANNOTATION_BIOTYPE_COL]):
            if pd.notna(symbol):
                key_to_biotype[str(symbol)] = str(biotype)

    selected = []
    rows = []
    for f in feature_cols:
        candidates = [str(f)]
        if GENE_ID_STRIP_VERSION_FOR_ANNOTATION:
            candidates.append(_strip_gene_version(f))
        biotype = None
        for k in candidates:
            if k in key_to_biotype:
                biotype = key_to_biotype[k]
                break
        keep = biotype in allowed
        if keep:
            selected.append(f)
        rows.append({"feature": f, "gene_biotype": biotype, "selected_by_biotype": bool(keep)})

    info = {
        "step": "gene_biotype_filter",
        "enabled": True,
        "gene_biotype_filter": str(GENE_BIOTYPE_FILTER),
        "allowed_biotypes": sorted([str(x) for x in allowed]),
        "n_before": int(len(feature_cols)),
        "n_after": int(len(selected)),
    }
    if len(selected) == 0:
        raise ValueError("gene biotype filter 이후 남은 feature가 없습니다. annotation 매칭 또는 GENE_BIOTYPE_FILTER를 확인하세요.")

    return selected, pd.DataFrame(rows), info


def _get_feature_matrix_for_selection(X_df, filtered_df, feature_cols):
    if X_df is not None:
        missing = [c for c in feature_cols if c not in X_df.columns]
        if len(missing) == 0:
            return X_df[feature_cols]
    missing = [c for c in feature_cols if c not in filtered_df.columns]
    if len(missing) > 0:
        raise KeyError(
            f"feature selection에 사용할 feature column 일부가 X_df/filtered_df에 없습니다. 예: {missing[:5]}"
        )
    return filtered_df[feature_cols]


def _get_y_for_feature_selection(y, filtered_df):
    if y is not None:
        return np.asarray(y)

    if DESEQ_DIAGNOSIS_COL not in filtered_df.columns:
        raise ValueError(
            "y가 전달되지 않았고 filtered_df에 DESEQ_DIAGNOSIS_COL이 없어 supervised feature selection을 수행할 수 없습니다."
        )
    labels = filtered_df[DESEQ_DIAGNOSIS_COL].astype(str).to_numpy()
    uniq = sorted(pd.unique(labels))
    mapping = {v: i for i, v in enumerate(uniq)}
    return np.array([mapping[v] for v in labels], dtype=np.int64)


def apply_expression_filter(X_train_df, feature_cols):
    if not bool(PREFILTER_EXPRESSION_ENABLED):
        return list(feature_cols), pd.DataFrame(), {
            "step": "expression_filter",
            "enabled": False,
            "n_before": int(len(feature_cols)),
            "n_after": int(len(feature_cols)),
        }

    arr = X_train_df[feature_cols].to_numpy(dtype=np.float32)
    frac = np.mean(arr > float(PREFILTER_EXPRESSION_MIN_VALUE), axis=0)
    keep_mask = frac >= float(PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC)
    selected = [f for f, keep in zip(feature_cols, keep_mask) if keep]

    score_df = pd.DataFrame({
        "feature": feature_cols,
        "expression_fraction": frac,
        "selected_by_expression": keep_mask,
    }).sort_values("expression_fraction", ascending=False)

    if len(selected) == 0:
        raise ValueError("expression pre-filter 이후 남은 feature가 없습니다. threshold를 낮추세요.")

    info = {
        "step": "expression_filter",
        "enabled": True,
        "min_value": float(PREFILTER_EXPRESSION_MIN_VALUE),
        "min_sample_frac": float(PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC),
        "n_before": int(len(feature_cols)),
        "n_after": int(len(selected)),
    }
    return selected, score_df, info


def apply_variance_filter(X_train_df, feature_cols):
    if PREFILTER_VARIANCE_TOP_N is None:
        return list(feature_cols), pd.DataFrame(), {
            "step": "variance_filter",
            "enabled": False,
            "n_before": int(len(feature_cols)),
            "n_after": int(len(feature_cols)),
        }

    top_n = int(PREFILTER_VARIANCE_TOP_N)
    if top_n <= 0:
        raise ValueError("PREFILTER_VARIANCE_TOP_N은 양수 또는 None이어야 합니다.")

    arr = X_train_df[feature_cols].to_numpy(dtype=np.float32)
    variances = np.nanvar(arr, axis=0)
    order = np.argsort(-variances)
    k = min(top_n, len(feature_cols))
    selected_idx = order[:k]
    selected = [feature_cols[i] for i in selected_idx]

    score_df = pd.DataFrame({
        "feature": feature_cols,
        "variance": variances,
    }).sort_values("variance", ascending=False)
    score_df["selected_by_variance"] = score_df["feature"].isin(selected)

    info = {
        "step": "variance_filter",
        "enabled": True,
        "top_n": int(top_n),
        "n_before": int(len(feature_cols)),
        "n_after": int(len(selected)),
    }
    return selected, score_df, info


def _rank_and_select(score_df, score_col, top_n=None, ascending=False):
    score_df = score_df.copy()
    score_df = score_df.sort_values(score_col, ascending=ascending)
    if top_n is None:
        selected = score_df["feature"].tolist()
    else:
        selected = score_df.head(int(top_n))["feature"].tolist()
    score_df["selected_by_feature_selection"] = score_df["feature"].isin(selected)
    return selected, score_df


def apply_supervised_feature_selection(fold, train_idx, filtered_df, X_train_df, y_train, candidate_features):
    method = str(FEATURE_SELECTION_METHOD).lower().strip()
    if method in ["none", "false", "off", "no", ""]:
        return list(candidate_features), pd.DataFrame(), {
            "step": "supervised_feature_selection",
            "enabled": False,
            "method": "none",
            "n_before": int(len(candidate_features)),
            "n_after": int(len(candidate_features)),
        }

    if method == "deseq":
        train_df = filtered_df.iloc[train_idx].reset_index(drop=True)
        deseq_df = run_or_load_fold_deseq(
            fold_train_df=train_df,
            fold=fold,
            use_existing=USE_EXISTING_DESEQ_RESULTS,
            run_if_missing=RUN_PYDESEQ2_IF_MISSING,
            counts_df=None,
        )

        selected, matched_deseq_df = get_deseq_selected_feature_cols(
            feature_cols=candidate_features,
            deseq_df=deseq_df,
            id_col="gene_id",
            pval_col=DESEQ_PVAL_COL,
            alpha=DESEQ_ALPHA,
            top_n=DESEQ_TOP_N if FEATURE_SELECTION_TOP_N is None else FEATURE_SELECTION_TOP_N,
            abs_lfc_min=DESEQ_ABS_LFC_MIN,
            strip_version=DESEQ_STRIP_VERSION,
            verbose=True,
        )
        info = {
            "step": "supervised_feature_selection",
            "enabled": True,
            "method": "deseq",
            "n_before": int(len(candidate_features)),
            "n_after": int(len(selected)),
            "deseq_result_path": str(get_deseq_result_path(fold)),
            "deseq_pval_col": DESEQ_PVAL_COL,
            "deseq_alpha": DESEQ_ALPHA,
            "deseq_top_n": DESEQ_TOP_N if FEATURE_SELECTION_TOP_N is None else FEATURE_SELECTION_TOP_N,
            "deseq_abs_lfc_min": DESEQ_ABS_LFC_MIN,
        }
        return selected, matched_deseq_df, info

    X_train = X_train_df[candidate_features].to_numpy(dtype=np.float32)
    y_train = np.asarray(y_train)

    if len(np.unique(y_train)) < 2:
        raise ValueError("train fold의 y class가 1개뿐이라 supervised feature selection을 수행할 수 없습니다.")

    if method in ["anova", "selectkbest", "f_classif"]:
        from sklearn.feature_selection import f_classif
        scores, pvals = f_classif(X_train, y_train)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
        pvals = np.nan_to_num(pvals, nan=1.0, posinf=1.0, neginf=1.0)
        score_df = pd.DataFrame({"feature": candidate_features, "score": scores, "pvalue": pvals})
        selected, score_df = _rank_and_select(score_df, "score", FEATURE_SELECTION_TOP_N, ascending=False)

    elif method in ["ttest", "t_test"]:
        if len(np.unique(y_train)) != 2:
            print("ttest는 binary task 전용이므로 multiclass에서는 ANOVA F-value로 대체합니다.")
            from sklearn.feature_selection import f_classif
            scores, pvals = f_classif(X_train, y_train)
            scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
            pvals = np.nan_to_num(pvals, nan=1.0, posinf=1.0, neginf=1.0)
            score_df = pd.DataFrame({"feature": candidate_features, "score": scores, "pvalue": pvals})
            selected, score_df = _rank_and_select(score_df, "score", FEATURE_SELECTION_TOP_N, ascending=False)
        else:
            from scipy.stats import ttest_ind
            classes = np.unique(y_train)
            X0 = X_train[y_train == classes[0]]
            X1 = X_train[y_train == classes[1]]
            stat, pvals = ttest_ind(X0, X1, axis=0, equal_var=False, nan_policy="omit")
            stat = np.nan_to_num(stat, nan=0.0, posinf=0.0, neginf=0.0)
            pvals = np.nan_to_num(pvals, nan=1.0, posinf=1.0, neginf=1.0)
            score_df = pd.DataFrame({
                "feature": candidate_features,
                "t_stat": stat,
                "abs_t_stat": np.abs(stat),
                "pvalue": pvals,
            })
            selected, score_df = _rank_and_select(score_df, "abs_t_stat", FEATURE_SELECTION_TOP_N, ascending=False)

    elif method in ["lasso", "l1"]:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                penalty="l1",
                C=float(FS_LINEAR_C),
                solver="liblinear",
                class_weight="balanced",
                max_iter=int(FS_LINEAR_MAX_ITER),
                random_state=RANDOM_STATE,
            )),
        ])
        clf.fit(X_train, y_train)
        coef = clf.named_steps["model"].coef_
        importance = np.mean(np.abs(coef), axis=0) if coef.ndim == 2 else np.abs(coef)
        score_df = pd.DataFrame({"feature": candidate_features, "importance": importance})
        nonzero = score_df.loc[score_df["importance"] > 0, "feature"].tolist()
        if FEATURE_SELECTION_TOP_N is None and len(nonzero) > 0:
            selected = nonzero
            score_df = score_df.sort_values("importance", ascending=False)
            score_df["selected_by_feature_selection"] = score_df["feature"].isin(selected)
        else:
            selected, score_df = _rank_and_select(score_df, "importance", FEATURE_SELECTION_TOP_N, ascending=False)

    elif method in ["elasticnet", "elastic_net"]:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                penalty="elasticnet",
                C=float(FS_LINEAR_C),
                l1_ratio=float(FS_ELASTICNET_L1_RATIO),
                solver="saga",
                class_weight="balanced",
                max_iter=int(FS_LINEAR_MAX_ITER),
                random_state=RANDOM_STATE,
                n_jobs=FS_TREE_N_JOBS,
            )),
        ])
        clf.fit(X_train, y_train)
        coef = clf.named_steps["model"].coef_
        importance = np.mean(np.abs(coef), axis=0) if coef.ndim == 2 else np.abs(coef)
        score_df = pd.DataFrame({"feature": candidate_features, "importance": importance})
        selected, score_df = _rank_and_select(score_df, "importance", FEATURE_SELECTION_TOP_N, ascending=False)

    elif method in ["rf_importance", "random_forest", "rf"]:
        from sklearn.ensemble import RandomForestClassifier

        clf = RandomForestClassifier(
            n_estimators=int(FS_TREE_N_ESTIMATORS),
            max_depth=FS_TREE_MAX_DEPTH,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=int(FS_TREE_N_JOBS),
        )
        clf.fit(X_train, y_train)
        score_df = pd.DataFrame({"feature": candidate_features, "importance": clf.feature_importances_})
        selected, score_df = _rank_and_select(score_df, "importance", FEATURE_SELECTION_TOP_N, ascending=False)

    elif method in ["xgb_importance", "xgboost", "xgb"]:
        try:
            from xgboost import XGBClassifier
        except Exception as e:
            raise ImportError("xgboost가 설치되어 있지 않아 xgb_importance feature selection을 수행할 수 없습니다.") from e

        clf = XGBClassifier(
            n_estimators=int(FS_TREE_N_ESTIMATORS),
            max_depth=FS_TREE_MAX_DEPTH if FS_TREE_MAX_DEPTH is not None else 4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=int(FS_TREE_N_JOBS),
        )
        from sklearn.utils.class_weight import compute_sample_weight
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
        clf.fit(X_train, y_train, sample_weight=sample_weight)
        score_df = pd.DataFrame({"feature": candidate_features, "importance": clf.feature_importances_})
        selected, score_df = _rank_and_select(score_df, "importance", FEATURE_SELECTION_TOP_N, ascending=False)

    else:
        raise ValueError(
            "FEATURE_SELECTION_METHOD는 'none', 'ttest', 'anova', 'deseq', 'lasso', "
            "'elasticnet', 'rf_importance', 'xgb_importance' 중 하나여야 합니다. "
            f"현재 값: {FEATURE_SELECTION_METHOD!r}"
        )

    if len(selected) == 0:
        raise ValueError(f"{method} feature selection 이후 남은 feature가 없습니다.")

    info = {
        "step": "supervised_feature_selection",
        "enabled": True,
        "method": method,
        "top_n": FEATURE_SELECTION_TOP_N,
        "n_before": int(len(candidate_features)),
        "n_after": int(len(selected)),
    }
    return selected, score_df, info


def select_features_for_fold(
    fold,
    train_idx,
    filtered_df,
    feature_cols,
    fold_dir=None,
    model_name=None,
    use_deseq=None,
    X_df=None,
    y=None,
):
    """
    하나의 fold에서 특정 모델이 사용할 feature column을 결정합니다.

    개선된 구조:
    1) annotation 기반 gene biotype filter
    2) train fold 기준 expression pre-filter
    3) train fold 기준 variance pre-filter
    4) train fold 기준 supervised feature selection

    기본 설정은 모두 비활성화되어 기존과 동일하게 전체 feature를 반환합니다.
    use_deseq 인자를 직접 True/False로 넘기면 기존 DESeq 전용 동작과 호환됩니다.
    """
    original_feature_cols = list(feature_cols)

    # backward compatibility: 기존 코드에서 use_deseq=True로 호출하면 method='deseq'로 강제합니다.
    effective_method = get_effective_feature_selection_method(model_name)
    if use_deseq is True:
        effective_method = "deseq"
    elif use_deseq is False:
        effective_method = "none"

    old_method = FEATURE_SELECTION_METHOD
    if effective_method != str(FEATURE_SELECTION_METHOD).lower().strip():
        # apply_supervised_feature_selection이 전역 설정을 읽는 구조이므로 임시로 맞춥니다.
        globals()["FEATURE_SELECTION_METHOD"] = effective_method

    fold_dir = Path(fold_dir) if fold_dir is not None else None
    if fold_dir is not None:
        fold_dir.mkdir(parents=True, exist_ok=True)

    try:
        selected_feature_cols = list(original_feature_cols)
        step_infos = []
        score_tables = {}

        X_source_df = _get_feature_matrix_for_selection(X_df, filtered_df, original_feature_cols)
        y_all_for_fs = _get_y_for_feature_selection(y, filtered_df)
        y_train = y_all_for_fs[train_idx]

        # 0. annotation 기반 biotype filter
        selected_feature_cols, biotype_df, info = apply_gene_biotype_filter(selected_feature_cols)
        step_infos.append(info)
        if fold_dir is not None and len(biotype_df) > 0:
            biotype_df.to_csv(fold_dir / "gene_biotype_filter_scores.csv", index=False)

        # 1. fold-wise expression filter
        X_train_df = X_source_df.iloc[train_idx]
        selected_feature_cols, expr_df, info = apply_expression_filter(X_train_df, selected_feature_cols)
        step_infos.append(info)
        if fold_dir is not None and len(expr_df) > 0:
            expr_df.to_csv(fold_dir / "expression_filter_scores.csv", index=False)

        # 2. fold-wise variance filter
        selected_feature_cols, var_df, info = apply_variance_filter(X_train_df, selected_feature_cols)
        step_infos.append(info)
        if fold_dir is not None and len(var_df) > 0:
            var_df.to_csv(fold_dir / "variance_filter_scores.csv", index=False)

        # pre-filtering 이후 feature 목록 저장
        if fold_dir is not None:
            pd.DataFrame({"feature": selected_feature_cols}).to_csv(
                fold_dir / "prefiltered_feature_columns.csv",
                index=False,
            )

        # 3. supervised feature selection
        fs_input_features = list(selected_feature_cols)
        selected_feature_cols, fs_scores_df, fs_info = apply_supervised_feature_selection(
            fold=fold,
            train_idx=train_idx,
            filtered_df=filtered_df,
            X_train_df=X_train_df,
            y_train=y_train,
            candidate_features=fs_input_features,
        )
        step_infos.append(fs_info)

        if fold_dir is not None:
            if len(fs_scores_df) > 0:
                fs_scores_df.to_csv(fold_dir / "feature_selection_scores.csv", index=False)
                # 기존 DESeq 파일명과도 호환되도록 별도 저장합니다.
                if effective_method == "deseq":
                    fs_scores_df.to_csv(fold_dir / "matched_deseq_features.csv", index=False)

            pd.DataFrame({"feature": selected_feature_cols}).to_csv(
                fold_dir / "selected_feature_columns.csv",
                index=False,
            )

            save_json(
                {
                    "model_name": model_name,
                    "feature_selection_label": get_feature_selection_label(model_name),
                    "gene_biotype_filter": str(GENE_BIOTYPE_FILTER),
                    "prefilter_expression_enabled": bool(PREFILTER_EXPRESSION_ENABLED),
                    "prefilter_expression_min_value": float(PREFILTER_EXPRESSION_MIN_VALUE),
                    "prefilter_expression_min_sample_frac": float(PREFILTER_EXPRESSION_MIN_SAMPLE_FRAC),
                    "prefilter_variance_top_n": None if PREFILTER_VARIANCE_TOP_N is None else int(PREFILTER_VARIANCE_TOP_N),
                    "feature_selection_method": effective_method,
                    "feature_selection_scope": FEATURE_SELECTION_SCOPE,
                    "feature_selection_top_n": FEATURE_SELECTION_TOP_N,
                    "legacy_use_deseq_feature_selection": bool(effective_method == "deseq"),
                    "legacy_deseq_feature_selection_scope": DESEQ_FEATURE_SELECTION_SCOPE,
                    "n_features_total": int(len(original_feature_cols)),
                    "n_features_after_prefilter": int(len(fs_input_features)),
                    "n_features_used": int(len(selected_feature_cols)),
                    "steps": step_infos,
                },
                fold_dir / "feature_selection_summary.json",
            )

        return selected_feature_cols, fs_scores_df if len(fs_scores_df) > 0 else None

    finally:
        globals()["FEATURE_SELECTION_METHOD"] = old_method

print("PYDESEQ2_AVAILABLE:", PYDESEQ2_AVAILABLE)
if should_use_deseq_for_any_model() and not PYDESEQ2_AVAILABLE:
    print("pyDESeq2 import 경고:", PYDESEQ2_IMPORT_ERROR)


## 6. TabNet 교차검증 실행

각 fold마다 다음을 저장합니다.

- 학습된 TabNet 모델
- fold별 평가 지표
- classification report
- confusion matrix
- prediction 결과
- feature importance
- 학습 history와 학습 곡선

`SCALE_TABNET_WITHIN_FOLD=True`로 설정하면 각 fold의 train split에만 `StandardScaler`를 fit한 뒤 valid split에 transform합니다.


In [ ]:

# 6. TabNet 교차검증 함수


def normalize_tabnet_train_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        "scratch_only": "scratch",
        "pretrain": "pretrained",
        "pretrained_only": "pretrained",
        "finetune": "pretrained",
        "fine_tune": "pretrained",
        "all": "both",
    }
    mode = aliases.get(mode, mode)
    if mode not in ["scratch", "pretrained", "both"]:
        raise ValueError("TABNET_TRAIN_MODE는 'scratch', 'pretrained', 'both' 중 하나여야 합니다.")
    return mode


def make_tabnet_classifier(seed):
    params = TABNET_MODEL_PARAMS.copy()
    params["seed"] = seed
    params["device_name"] = "auto"
    return TabNetClassifier(**params)


def get_pretrained_fold_dir(fold):
    if PRETRAINED_TABNET_RUN_DIR is None:
        raise ValueError(
            "pretrained fine-tuning을 사용하려면 PRETRAINED_TABNET_RUN_DIR에 "
            "사전학습 결과 전체 run directory를 지정해야 합니다."
        )
    return Path(PRETRAINED_TABNET_RUN_DIR) / f"fold_{fold:02d}"


def get_pretrained_model_path(fold):
    fold_dir = get_pretrained_fold_dir(fold)
    model_path = fold_dir / PRETRAINED_MODEL_FILENAME

    # pytorch-tabnet save_model/load_model 버전 차이를 고려한 fallback
    candidates = [
        model_path,
        Path(str(model_path).removesuffix(".zip")),
        fold_dir / "tabnet_pretrainer.zip",
        fold_dir / "tabnet_pretrainer",
        fold_dir / "tabnet_pretrainer_full.zip",
        fold_dir / "tabnet_pretrainer_full",
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        "fold별 pretrained TabNetPretrainer 파일을 찾지 못했습니다. 확인한 후보:"
        + "\n" + "\n".join(str(c) for c in candidates)
    )


def load_pretrained_tabnet_pretrainer(fold):
    model_path = get_pretrained_model_path(fold)
    pretrainer = TabNetPretrainer()
    pretrainer.load_model(str(model_path))
    print(f"[Pretrained] fold {fold} encoder loaded from: {model_path}")
    return pretrainer, model_path


def load_pretrained_feature_columns(fold):
    path = get_pretrained_fold_dir(fold) / PRETRAINED_FEATURE_FILENAME
    if not path.exists():
        raise FileNotFoundError(f"pretrained selected feature 파일이 없습니다: {path}")
    df = pd.read_csv(path)
    if "feature" not in df.columns:
        raise ValueError(f"{path} 파일에 'feature' 컬럼이 없습니다.")
    features = df["feature"].dropna().astype(str).tolist()
    if len(features) == 0:
        raise ValueError(f"pretrained selected feature가 비어 있습니다: {path}")
    return features, path


def load_pretrained_scaler_if_available(fold):
    path = get_pretrained_fold_dir(fold) / PRETRAINED_SCALER_FILENAME
    if USE_PRETRAINED_SCALER and path.exists():
        print(f"[Pretrained] fold {fold} scaler loaded from: {path}")
        return joblib.load(path), path
    if USE_PRETRAINED_SCALER:
        print(f"[Pretrained] fold {fold} scaler 파일이 없어 사용하지 않습니다: {path}")
    return None, None


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass



def _count_trainable_params(module):
    if module is None:
        return {"total": 0, "trainable": 0, "frozen": 0}
    total = sum(int(p.numel()) for p in module.parameters())
    trainable = sum(int(p.numel()) for p in module.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}


def _find_tabnet_encoder_and_embedder(network):
    """pytorch-tabnet 버전별 network 구조 차이를 고려해 encoder/embedder/head 후보를 찾습니다."""
    encoder = None
    embedder = None
    head = None

    if network is None:
        return encoder, embedder, head

    # 일반적인 TabNetClassifier 구조: network.embedder + network.tabnet.encoder/final_mapping
    if hasattr(network, "tabnet"):
        tabnet = network.tabnet
        if hasattr(tabnet, "encoder"):
            encoder = tabnet.encoder
        if hasattr(tabnet, "final_mapping"):
            head = tabnet.final_mapping
        elif hasattr(tabnet, "final_layer"):
            head = tabnet.final_layer

    # 일부 버전 또는 pretrainer 계열 구조 fallback
    if encoder is None and hasattr(network, "encoder"):
        encoder = network.encoder
    if embedder is None and hasattr(network, "embedder"):
        embedder = network.embedder
    if embedder is None and hasattr(network, "tabnet") and hasattr(network.tabnet, "embedder"):
        embedder = network.tabnet.embedder
    if head is None:
        for candidate in ["final_mapping", "final_layer", "classifier", "fc"]:
            if hasattr(network, candidate):
                head = getattr(network, candidate)
                break

    return encoder, embedder, head


def set_pretrained_encoder_trainability(network, freeze_encoder=False, freeze_embedder=True, verbose=True):
    """
    pretrained TabNetClassifier 학습 시작 시 Encoder freeze 여부를 명시적으로 적용합니다.

    - freeze_encoder=False: Encoder를 포함한 전체 network를 trainable 상태로 둡니다.
    - freeze_encoder=True: encoder module의 requires_grad=False, classification head는 trainable 유지.
    """
    if network is None:
        warnings.warn("TabNet network가 아직 생성되지 않아 Encoder trainability를 설정하지 못했습니다.")
        return {"applied": False, "reason": "network_is_none"}

    encoder, embedder, head = _find_tabnet_encoder_and_embedder(network)

    # 먼저 전체 network를 trainable로 복원해 두고, 필요한 부분만 freeze합니다.
    for p in network.parameters():
        p.requires_grad = True

    frozen_modules = []
    if freeze_encoder:
        if encoder is None:
            warnings.warn("TabNet encoder module을 찾지 못해 freeze를 적용하지 못했습니다.")
        else:
            for p in encoder.parameters():
                p.requires_grad = False
            frozen_modules.append("encoder")

        if freeze_embedder and embedder is not None:
            for p in embedder.parameters():
                p.requires_grad = False
            frozen_modules.append("embedder")

    info = {
        "applied": True,
        "freeze_encoder": bool(freeze_encoder),
        "freeze_embedder": bool(freeze_embedder),
        "frozen_modules": frozen_modules,
        "network_params": _count_trainable_params(network),
        "encoder_params": _count_trainable_params(encoder),
        "embedder_params": _count_trainable_params(embedder),
        "head_params": _count_trainable_params(head),
        "encoder_found": encoder is not None,
        "embedder_found": embedder is not None,
        "head_found": head is not None,
    }

    if verbose:
        print("[TabNet trainability]", json.dumps(info, indent=2, ensure_ascii=False))

    return info


class TabNetEncoderTrainabilityCallback:
    """pytorch-tabnet fit() 내부에서 network 생성 및 from_unsupervised 로딩 후 freeze 설정을 적용하기 위한 callback."""

    def __init__(self, freeze_encoder=False, freeze_embedder=True, verbose=True):
        self.freeze_encoder = bool(freeze_encoder)
        self.freeze_embedder = bool(freeze_embedder)
        self.verbose = bool(verbose)
        self.trainer = None
        self.trainability_info = None

    def set_trainer(self, trainer):
        self.trainer = trainer

    def set_params(self, params):
        self.params = params

    def on_train_begin(self, logs=None):
        network = getattr(self.trainer, "network", None)
        self.trainability_info = set_pretrained_encoder_trainability(
            network=network,
            freeze_encoder=self.freeze_encoder,
            freeze_embedder=self.freeze_embedder,
            verbose=self.verbose,
        )

    # pytorch-tabnet CallbackContainer 호환용 no-op 메서드들
    def on_train_end(self, logs=None):
        pass

    def on_epoch_begin(self, epoch, logs=None):
        pass

    def on_epoch_end(self, epoch, logs=None):
        pass

    def on_batch_begin(self, batch, logs=None):
        pass

    def on_batch_end(self, batch, logs=None):
        pass

def fit_one_tabnet_classifier(
    model_name,
    X_train,
    y_train,
    X_valid,
    y_valid,
    seed,
    from_unsupervised=None,
    freeze_pretrained_encoder=False,
    freeze_pretrained_embedder=True,
):
    clf = make_tabnet_classifier(seed=seed)

    fit_params = TABNET_FIT_PARAMS.copy()
    callbacks = list(fit_params.pop("callbacks", []) or [])
    trainability_callback = None

    fit_kwargs = {}
    if from_unsupervised is not None:
        fit_kwargs["from_unsupervised"] = from_unsupervised
        trainability_callback = TabNetEncoderTrainabilityCallback(
            freeze_encoder=freeze_pretrained_encoder,
            freeze_embedder=freeze_pretrained_embedder,
            verbose=PRINT_PRETRAINED_ENCODER_TRAINABILITY,
        )
        callbacks.append(trainability_callback)

    if callbacks:
        fit_params["callbacks"] = callbacks

    clf.fit(
        X_train=X_train,
        y_train=y_train,
        eval_set=[(X_valid, y_valid)],
        eval_name=["valid"],
        **fit_params,
        **fit_kwargs,
    )

    if trainability_callback is not None:
        clf.pretrained_trainability_info = trainability_callback.trainability_info

    return clf


def prepare_tabnet_fold_arrays(
    X_df,
    train_idx,
    valid_idx,
    selected_feature_cols,
    fold_dir,
    scale_within_fold=False,
    pretrained_scaler=None,
):
    X_train = X_df.iloc[train_idx][selected_feature_cols].to_numpy(dtype=np.float32)
    X_valid = X_df.iloc[valid_idx][selected_feature_cols].to_numpy(dtype=np.float32)
    X_train = np.ascontiguousarray(X_train, dtype=np.float32)
    X_valid = np.ascontiguousarray(X_valid, dtype=np.float32)

    scaler = None
    if pretrained_scaler is not None:
        X_train = pretrained_scaler.transform(X_train).astype(np.float32)
        X_valid = pretrained_scaler.transform(X_valid).astype(np.float32)
        X_train = np.ascontiguousarray(X_train, dtype=np.float32)
        X_valid = np.ascontiguousarray(X_valid, dtype=np.float32)
        scaler = pretrained_scaler
    elif scale_within_fold:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train).astype(np.float32)
        X_valid = scaler.transform(X_valid).astype(np.float32)
        X_train = np.ascontiguousarray(X_train, dtype=np.float32)
        X_valid = np.ascontiguousarray(X_valid, dtype=np.float32)
        joblib.dump(scaler, Path(fold_dir) / "scaler.joblib")

    return X_train, X_valid, scaler


def evaluate_save_tabnet_model(
    clf,
    model_name,
    fold,
    model_dir,
    X_valid,
    y_valid,
    valid_idx,
    selected_feature_cols,
    feature_cols,
    feature_selection_label,
    pretrained_model_path=None,
    pretrained_encoder_frozen=None,
):
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)

    # predict_proba 한 번으로 y_pred까지 만들면 validation inference 중복을 줄일 수 있습니다.
    y_score = None
    try:
        y_score = clf.predict_proba(X_valid)
        y_pred = np.argmax(y_score, axis=1)
    except Exception:
        y_pred = clf.predict(X_valid)
        y_score = None

    metric = save_evaluation_outputs(
        model_name=model_name,
        fold=fold,
        fold_dir=model_dir,
        y_true=y_valid,
        y_pred=y_pred,
        y_score=y_score,
        eval_indices=valid_idx,
    )
    metric["n_features_used"] = int(len(selected_feature_cols))
    metric["n_features_total"] = int(len(feature_cols))
    metric["feature_selection"] = feature_selection_label
    metric["pretrained_model_path"] = str(pretrained_model_path) if pretrained_model_path is not None else None
    metric["pretrained_encoder_frozen"] = pretrained_encoder_frozen

    saved_file = clf.save_model(str(model_dir / "tabnet_model"))
    print("Saved model:", saved_file)

    fi_df = save_feature_importance(
        importance=getattr(clf, "feature_importances_", None),
        feature_names=selected_feature_cols,
        out_path=model_dir / "feature_importance.csv",
    )
    plot_and_save_top_importance(
        fi_df,
        out_path=model_dir / "feature_importance_top30.png",
        title=f"{model_name} Fold {fold} Top Feature Importance",
        top_n=30,
    )

    if getattr(clf, "pretrained_trainability_info", None) is not None:
        save_json(clf.pretrained_trainability_info, model_dir / "pretrained_trainability_info.json")

    save_json(
        {
            "model_name": model_name,
            "fold": int(fold),
            "n_features_used": int(len(selected_feature_cols)),
            "feature_selection": feature_selection_label,
            "pretrained_model_path": str(pretrained_model_path) if pretrained_model_path is not None else None,
            "pretrained_encoder_frozen": pretrained_encoder_frozen,
            "pretrained_trainability_info": getattr(clf, "pretrained_trainability_info", None),
        },
        model_dir / "model_run_summary.json",
    )

    return metric, fi_df


def run_tabnet_cv(
    X_df,
    y,
    groups,
    filtered_df,
    feature_cols,
    n_splits=5,
    random_state=42,
    out_dir=RUN_DIR / "tabnet_cv",
    scale_within_fold=False,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    X_df = X_df.reset_index(drop=True)
    filtered_df = filtered_df.reset_index(drop=True)

    train_mode = normalize_tabnet_train_mode(TABNET_TRAIN_MODE)
    tabnet_model_names = []
    if train_mode in ["scratch", "both"]:
        tabnet_model_names.append("tabnet_scratch")
    if train_mode in ["pretrained", "both"]:
        tabnet_model_names.append("tabnet_pretrained_finetune")

    save_json(
        {
            "tabnet_train_mode": train_mode,
            "tabnet_model_names": tabnet_model_names,
            "pretrained_tabnet_run_dir": str(PRETRAINED_TABNET_RUN_DIR) if PRETRAINED_TABNET_RUN_DIR is not None else None,
            "use_pretrained_features": bool(USE_PRETRAINED_FEATURES),
            "strict_pretrained_feature_match": bool(STRICT_PRETRAINED_FEATURE_MATCH),
            "use_pretrained_scaler": bool(USE_PRETRAINED_SCALER),
            "freeze_pretrained_encoder": bool(FREEZE_PRETRAINED_ENCODER),
            "freeze_pretrained_embedder": bool(FREEZE_PRETRAINED_EMBEDDER),
            "deseq_feature_selection_scope": DESEQ_FEATURE_SELECTION_SCOPE,
            "deseq_feature_selection_models": DESEQ_FEATURE_SELECTION_MODELS,
        },
        out_dir / "tabnet_cv_config.json",
    )

    splitter = get_cv_splitter(n_splits=n_splits, random_state=random_state, use_groups=True)
    metrics = []
    fold_importance_list = []

    split_iter = splitter.split(X_df, y, groups=groups)

    for fold, (train_idx, valid_idx) in enumerate(split_iter, start=1):
        print("=" * 80)
        print(f"[TabNet] Fold {fold}/{n_splits}")
        print("train samples:", len(train_idx), "valid samples:", len(valid_idx))
        print("train patients:", len(np.unique(groups[train_idx])), "valid patients:", len(np.unique(groups[valid_idx])))

        patient_overlap = set(groups[train_idx]) & set(groups[valid_idx])
        print("patient overlap:", len(patient_overlap))
        if len(patient_overlap) > 0:
            raise RuntimeError("Train/valid fold 사이에 동일 환자가 중복되었습니다.")

        fold_root_dir = out_dir / f"fold_{fold:02d}"
        fold_root_dir.mkdir(parents=True, exist_ok=True)

        y_train, y_valid = y[train_idx], y[valid_idx]

        for model_name in tabnet_model_names:
            clf = None
            pretrainer = None
            scaler = None
            X_train = X_valid = None
            history_df = fi_df = tmp = None
            pretrained_model_path = None

            model_dir = fold_root_dir / model_name
            model_dir.mkdir(parents=True, exist_ok=True)

            try:
                print("-" * 80)
                print(f"[{model_name}] Fold {fold}/{n_splits}")

                if model_name == "tabnet_pretrained_finetune":
                    pretrained_features, pretrained_feature_path = load_pretrained_feature_columns(fold)
                    if USE_PRETRAINED_FEATURES:
                        selected_feature_cols = pretrained_features
                        feature_selection_label = "pretrained_features"
                        pd.DataFrame({"feature": selected_feature_cols}).to_csv(
                            model_dir / "selected_feature_columns.csv",
                            index=False,
                        )
                        save_json(
                            {
                                "model_name": model_name,
                                "use_pretrained_features": True,
                                "pretrained_feature_path": str(pretrained_feature_path),
                                "n_features_total": int(len(feature_cols)),
                                "n_features_used": int(len(selected_feature_cols)),
                                "note": "pretrained encoder 입력 차원/순서를 맞추기 위해 사전학습 feature list를 사용했습니다.",
                            },
                            model_dir / "feature_selection_summary.json",
                        )
                    else:
                        selected_feature_cols, _ = select_features_for_fold(
                            fold=fold,
                            train_idx=train_idx,
                            filtered_df=filtered_df,
                            X_df=X_df,
                            y=y,
                            feature_cols=feature_cols,
                            fold_dir=model_dir,
                            model_name=model_name,
                        )
                        feature_selection_label = get_feature_selection_label(model_name)
                        if STRICT_PRETRAINED_FEATURE_MATCH and selected_feature_cols != pretrained_features:
                            raise ValueError(
                                "pretrained fine-tuning 입력 feature가 사전학습 feature와 다릅니다. "
                                "USE_PRETRAINED_FEATURES=True로 두거나, 사전학습과 동일한 feature set을 사용하세요."
                            )

                    missing = [c for c in selected_feature_cols if c not in X_df.columns]
                    if missing:
                        raise ValueError(
                            f"pretrained feature 중 downstream X_df에 없는 feature가 있습니다. 예: {missing[:10]}"
                        )

                    pretrainer, pretrained_model_path = load_pretrained_tabnet_pretrainer(fold)
                    pretrained_scaler, pretrained_scaler_path = load_pretrained_scaler_if_available(fold)
                    X_train, X_valid, scaler = prepare_tabnet_fold_arrays(
                        X_df=X_df,
                        train_idx=train_idx,
                        valid_idx=valid_idx,
                        selected_feature_cols=selected_feature_cols,
                        fold_dir=model_dir,
                        scale_within_fold=False if pretrained_scaler is not None else scale_within_fold,
                        pretrained_scaler=pretrained_scaler,
                    )
                else:
                    selected_feature_cols, _ = select_features_for_fold(
                        fold=fold,
                        train_idx=train_idx,
                        filtered_df=filtered_df,
                        X_df=X_df,
                        y=y,
                        feature_cols=feature_cols,
                        fold_dir=model_dir,
                        model_name=model_name,
                    )
                    feature_selection_label = get_feature_selection_label(model_name)
                    X_train, X_valid, scaler = prepare_tabnet_fold_arrays(
                        X_df=X_df,
                        train_idx=train_idx,
                        valid_idx=valid_idx,
                        selected_feature_cols=selected_feature_cols,
                        fold_dir=model_dir,
                        scale_within_fold=scale_within_fold,
                        pretrained_scaler=None,
                    )

                print("features used:", len(selected_feature_cols), "/", len(feature_cols))

                pretrained_encoder_frozen = bool(FREEZE_PRETRAINED_ENCODER) if model_name == "tabnet_pretrained_finetune" else None

                clf = fit_one_tabnet_classifier(
                    model_name=model_name,
                    X_train=X_train,
                    y_train=y_train,
                    X_valid=X_valid,
                    y_valid=y_valid,
                    seed=random_state + fold,
                    from_unsupervised=pretrainer if model_name == "tabnet_pretrained_finetune" else None,
                    freeze_pretrained_encoder=bool(FREEZE_PRETRAINED_ENCODER) if model_name == "tabnet_pretrained_finetune" else False,
                    freeze_pretrained_embedder=bool(FREEZE_PRETRAINED_EMBEDDER),
                )

                history_df = save_tabnet_history(clf.history.history, model_dir)

                metric, fi_df = evaluate_save_tabnet_model(
                    clf=clf,
                    model_name=model_name,
                    fold=fold,
                    model_dir=model_dir,
                    X_valid=X_valid,
                    y_valid=y_valid,
                    valid_idx=valid_idx,
                    selected_feature_cols=selected_feature_cols,
                    feature_cols=feature_cols,
                    feature_selection_label=feature_selection_label,
                    pretrained_model_path=pretrained_model_path,
                    pretrained_encoder_frozen=pretrained_encoder_frozen,
                )
                metrics.append(metric)

                if fi_df is not None:
                    tmp = fi_df.copy()
                    tmp["model"] = model_name
                    tmp["fold"] = fold
                    fold_importance_list.append(tmp)

                print("Fold metric:", metric)
            finally:
                try:
                    if clf is not None and hasattr(clf, "network") and clf.network is not None:
                        clf.network.to("cpu")
                except Exception as e:
                    print("TabNet classifier network CPU 이동 중 경고:", e)

                try:
                    if pretrainer is not None and hasattr(pretrainer, "network") and pretrainer.network is not None:
                        pretrainer.network.to("cpu")
                except Exception as e:
                    print("TabNet pretrainer network CPU 이동 중 경고:", e)

                del clf, pretrainer, scaler
                del X_train, X_valid, history_df, fi_df, tmp
                clear_cuda_memory()

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(out_dir / "tabnet_cv_metrics.csv", index=False)

    summary_df = summarize_metrics(metrics_df, group_cols=("model",)) if len(metrics_df) > 0 else pd.DataFrame()
    summary_df.to_csv(out_dir / "tabnet_cv_metrics_summary.csv", index=False)

    if fold_importance_list:
        all_fi_df = pd.concat(fold_importance_list, ignore_index=True)
        all_fi_df.to_csv(out_dir / "tabnet_feature_importance_all_folds.csv", index=False)

        fi_summary = (
            all_fi_df
            .groupby(["model", "feature"])["importance"]
            .agg(["mean", "std", "min", "max"])
            .sort_values(["model", "mean"], ascending=[True, False])
            .reset_index()
        )
        fi_summary.to_csv(out_dir / "tabnet_feature_importance_summary.csv", index=False)

        for model_name, sub in fi_summary.groupby("model"):
            plot_and_save_top_importance(
                sub.rename(columns={"mean": "importance"})[["feature", "importance"]],
                out_path=out_dir / f"{model_name}_feature_importance_mean_top30.png",
                title=f"{model_name} Mean Feature Importance Across Folds",
                top_n=30,
            )

    print("=" * 80)
    print("[TabNet CV Summary]")
    display(summary_df)

    return metrics_df, summary_df


In [ ]:
# 6-1. TabNet 교차검증 실행

tabnet_metrics_df = pd.DataFrame()
tabnet_summary_df = pd.DataFrame()

if RUN_TABNET_CV:
    tabnet_metrics_df, tabnet_summary_df = run_tabnet_cv(
        X_df=X_all_df,
        y=y_all,
        groups=groups_all,
        filtered_df=filtered_df,
        feature_cols=feature_cols_tabnet,
        n_splits=N_SPLITS,
        random_state=RANDOM_STATE,
        out_dir=RUN_DIR / "tabnet_cv",
        scale_within_fold=SCALE_TABNET_WITHIN_FOLD,
    )
else:
    print("RUN_TABNET_CV=False이므로 TabNet 교차검증을 건너뜁니다.")


## 7. Baseline 모델 교차검증

`BASELINE_MODELS`가 `None`이면 baseline은 실행하지 않습니다.

지원하는 baseline 이름은 다음과 같습니다.

- `"svm_linear"`: `StandardScaler + LinearSVC`
- `"rf"`: `RandomForestClassifier`
- `"logreg"`: `StandardScaler + LogisticRegression`
- `"xgboost"`: `XGBClassifier`, `xgboost` 설치 필요
- `"all"`: 사용 가능한 모든 baseline 실행

각 baseline도 TabNet과 동일하게 fold별 평가 결과, confusion matrix, prediction, 가능한 경우 feature importance를 저장합니다.


In [ ]:
# 7. Baseline 모델 함수

def normalize_baseline_models(baseline_models):
    if baseline_models is None:
        return []

    if isinstance(baseline_models, str):
        if baseline_models.lower() == "all":
            return ["svm_linear", "rf", "logreg", "xgboost"]
        return [baseline_models.lower()]

    return [str(m).lower() for m in baseline_models]


def make_baseline_model(model_name, random_state=42):
    model_name = model_name.lower()

    if model_name == "svm_linear":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=random_state,
            )),
        ])

    if model_name == "rf":
        return RandomForestClassifier(
            n_estimators=30,
            max_depth=8,
            min_samples_leaf=2,
            max_leaf_nodes=4,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=random_state,
        )

    if model_name == "logreg":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                penalty="l2",
                C=1.0,
                class_weight="balanced",
                solver="saga",
                max_iter=3000,
                n_jobs=-1,
                random_state=random_state,
            )),
        ])

    if model_name == "xgboost":
        try:
            from xgboost import XGBClassifier
        except Exception as e:
            raise ImportError(
                "xgboost가 설치되어 있지 않습니다. "
                "설치 후 다시 실행하거나 BASELINE_MODELS에서 'xgboost'를 제거하세요."
            ) from e

        return XGBClassifier(
            n_estimators=70,
            max_depth=5,
            max_leaves=5,
            learning_rate=0.2,
            subsample=0.8,
            colsample_bytree=0.5,
            objective="binary:logistic" if len(class_labels) == 2 else "multi:softprob",
            eval_metric="logloss",
            tree_method="hist",
            random_state=random_state,
            n_jobs=-1,
        )

    raise ValueError(f"지원하지 않는 baseline 모델입니다: {model_name}")


def get_model_scores(model, X_valid):
    """
    predict_proba가 있으면 probability를 저장하고,
    없으면 decision_function 값을 저장합니다.
    """
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X_valid)
        except Exception:
            pass

    if hasattr(model, "decision_function"):
        try:
            return model.decision_function(X_valid)
        except Exception:
            pass

    return None


def extract_baseline_feature_importance(model):
    """
    모델이 제공하는 feature importance 또는 coefficient를 추출합니다.
    Pipeline인 경우 마지막 step의 model을 확인합니다.
    """
    estimator = model
    if isinstance(model, Pipeline):
        estimator = model.named_steps.get("model", model.steps[-1][1])

    if hasattr(estimator, "feature_importances_"):
        return np.asarray(estimator.feature_importances_)

    if hasattr(estimator, "coef_"):
        coef = np.asarray(estimator.coef_)
        if coef.ndim == 1:
            return np.abs(coef)
        return np.mean(np.abs(coef), axis=0)

    return None


def run_baseline_cv(
    X_df,
    y,
    groups,
    filtered_df,
    feature_cols,
    baseline_models=None,
    n_splits=5,
    random_state=42,
    out_dir=RUN_DIR / "baseline_cv",
):
    baseline_models = normalize_baseline_models(baseline_models)

    if len(baseline_models) == 0:
        print("BASELINE_MODELS=None 또는 [] 이므로 baseline 교차검증을 실행하지 않습니다.")
        return pd.DataFrame(), pd.DataFrame()

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    X_df = X_df.reset_index(drop=True)
    filtered_df = filtered_df.reset_index(drop=True)

    all_metrics = []

    for model_name in baseline_models:
        print("=" * 80)
        print(f"[Baseline] {model_name}")

        model_out_dir = out_dir / model_name
        model_out_dir.mkdir(parents=True, exist_ok=True)

        splitter = get_cv_splitter(n_splits=n_splits, random_state=random_state, use_groups=True)

        fold_importance_list = []

        for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_df, y, groups=groups), start=1):
            print("-" * 80)
            print(f"[{model_name}] Fold {fold}/{n_splits}")
            print("train samples:", len(train_idx), "valid samples:", len(valid_idx))
            print("train patients:", len(np.unique(groups[train_idx])), "valid patients:", len(np.unique(groups[valid_idx])))

            patient_overlap = set(groups[train_idx]) & set(groups[valid_idx])
            print("patient overlap:", len(patient_overlap))
            if len(patient_overlap) > 0:
                raise RuntimeError("Train/valid fold 사이에 동일 환자가 중복되었습니다.")

            fold_dir = model_out_dir / f"fold_{fold:02d}"
            fold_dir.mkdir(parents=True, exist_ok=True)

            selected_feature_cols, _ = select_features_for_fold(
                fold=fold,
                train_idx=train_idx,
                filtered_df=filtered_df,
                X_df=X_df,
                y=y,
                feature_cols=feature_cols,
                fold_dir=fold_dir,
                model_name=model_name,
            )

            print("features used:", len(selected_feature_cols), "/", len(feature_cols))

            X_train = X_df.iloc[train_idx][selected_feature_cols].to_numpy(dtype=np.float32)
            X_valid = X_df.iloc[valid_idx][selected_feature_cols].to_numpy(dtype=np.float32)
            y_train, y_valid = y[train_idx], y[valid_idx]

            try:
                model = make_baseline_model(model_name, random_state=random_state + fold)
                model.fit(X_train, y_train)
            except Exception as e:
                print(f"{model_name} fold {fold} 학습 실패:", e)
                error_info = {
                    "model": model_name,
                    "fold": fold,
                    "error": str(e),
                    "n_features_used": int(len(selected_feature_cols)),
                }
                save_json(error_info, fold_dir / "error.json")
                continue

            y_pred = model.predict(X_valid)
            y_score = get_model_scores(model, X_valid)

            metric = save_evaluation_outputs(
                model_name=model_name,
                fold=fold,
                fold_dir=fold_dir,
                y_true=y_valid,
                y_pred=y_pred,
                y_score=y_score,
                eval_indices=valid_idx,
            )
            metric["n_features_used"] = int(len(selected_feature_cols))
            metric["n_features_total"] = int(len(feature_cols))
            metric["feature_selection"] = get_feature_selection_label(model_name)
            all_metrics.append(metric)

            joblib.dump(model, fold_dir / f"{model_name}.joblib")

            importance = extract_baseline_feature_importance(model)
            fi_df = save_feature_importance(
                importance=importance,
                feature_names=selected_feature_cols,
                out_path=fold_dir / "feature_importance.csv",
            )
            plot_and_save_top_importance(
                fi_df,
                out_path=fold_dir / "feature_importance_top30.png",
                title=f"{model_name} Fold {fold} Top Feature Importance",
                top_n=30,
            )

            if fi_df is not None:
                tmp = fi_df.copy()
                tmp["model"] = model_name
                tmp["fold"] = fold
                fold_importance_list.append(tmp)

            print("Fold metric:", metric)

        if fold_importance_list:
            all_fi_df = pd.concat(fold_importance_list, ignore_index=True)
            all_fi_df.to_csv(model_out_dir / f"{model_name}_feature_importance_all_folds.csv", index=False)

            fi_summary = (
                all_fi_df
                .groupby("feature")["importance"]
                .agg(["mean", "std", "min", "max"])
                .sort_values("mean", ascending=False)
                .reset_index()
            )
            fi_summary.to_csv(model_out_dir / f"{model_name}_feature_importance_summary.csv", index=False)

            plot_and_save_top_importance(
                fi_summary.rename(columns={"mean": "importance"})[["feature", "importance"]],
                out_path=model_out_dir / f"{model_name}_feature_importance_mean_top30.png",
                title=f"{model_name} Mean Feature Importance Across Folds",
                top_n=30,
            )

    metrics_df = pd.DataFrame(all_metrics)
    if len(metrics_df) > 0:
        metrics_df.to_csv(out_dir / "baseline_cv_metrics.csv", index=False)
        summary_df = summarize_metrics(metrics_df, group_cols=("model",))
        summary_df.to_csv(out_dir / "baseline_cv_metrics_summary.csv", index=False)
    else:
        summary_df = pd.DataFrame()

    print("=" * 80)
    print("[Baseline CV Summary]")
    display(summary_df)

    return metrics_df, summary_df


In [ ]:
# 7-1. Baseline 교차검증 실행

baseline_metrics_df, baseline_summary_df = run_baseline_cv(
    X_df=X_all_df,
    y=y_all,
    groups=groups_all,
    filtered_df=filtered_df,
    feature_cols=feature_cols_tabnet,
    baseline_models=BASELINE_MODELS,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    out_dir=RUN_DIR / "baseline_cv",
)


## 8. 전체 결과 통합

TabNet과 baseline 결과를 하나로 합쳐 저장합니다.  
여기서 생성된 `all_cv_metrics.csv`와 `all_cv_metrics_summary.csv`를 보고서 작성에 바로 사용할 수 있습니다.


In [ ]:
# 8. 전체 결과 통합 저장

metrics_list = []

if "tabnet_metrics_df" in globals() and len(tabnet_metrics_df) > 0:
    metrics_list.append(tabnet_metrics_df)

if "baseline_metrics_df" in globals() and len(baseline_metrics_df) > 0:
    metrics_list.append(baseline_metrics_df)

if metrics_list:
    all_metrics_df = pd.concat(metrics_list, ignore_index=True)
    all_metrics_df.to_csv(RUN_DIR / "all_cv_metrics.csv", index=False)

    all_summary_df = summarize_metrics(all_metrics_df, group_cols=("model",))
    all_summary_df.to_csv(RUN_DIR / "all_cv_metrics_summary.csv", index=False)

    print("[All CV Metrics]")
    display(all_metrics_df)

    print("[All CV Summary]")
    display(all_summary_df)
else:
    all_metrics_df = pd.DataFrame()
    all_summary_df = pd.DataFrame()
    print("저장할 metric 결과가 없습니다.")

print("모든 결과 저장 위치:", RUN_DIR.resolve())


## 9. 사용 방법 요약

### DESeq feature selection을 끄고 싶은 경우

설정 셀에서 다음처럼 수정합니다.

```python
USE_DESEQ_FEATURE_SELECTION = False
```

### 기존 fold별 DESeq 결과만 사용하고 싶은 경우

```python
USE_DESEQ_FEATURE_SELECTION = True
USE_EXISTING_DESEQ_RESULTS = True
RUN_PYDESEQ2_IF_MISSING = False
DESEQ_ON_MISSING = "error"
```

외부 경로에 이미 저장된 결과를 사용하려면 다음처럼 지정합니다.

```python
EXTERNAL_DESEQ_RESULT_TEMPLATE = Path("./pydeseq2_cv_results/task1/task1_fold{fold}_deseq.csv")
```

또는 `{fold_name}`을 사용할 수 있습니다.

```python
EXTERNAL_DESEQ_RESULT_TEMPLATE = Path("./pydeseq2_cv_results/{fold_name}_deseq.csv")
```

### DEG 선별 기준을 조정하고 싶은 경우

```python
DESEQ_PVAL_COL = "padj"
DESEQ_ALPHA = 0.05
DESEQ_TOP_N = 1000
DESEQ_ABS_LFC_MIN = 0.5
```

### baseline 모델을 추가로 실행하고 싶은 경우

위의 설정 셀에서 다음처럼 수정한 뒤, 7번 baseline 교차검증 셀부터 다시 실행하면 됩니다.

```python
BASELINE_MODELS = ["svm_linear", "rf", "xgboost"]
```

### fold 수를 바꾸고 싶은 경우

```python
N_SPLITS = 10
```

### 저장 결과 확인

실행 후 `RUN_DIR` 아래에 다음 구조로 결과가 저장됩니다.

```text
results_tabnet_rna4/
  pd_control_tabnet_cv_deseq_YYYYMMDD_HHMMSS/
    dataset_summary.json
    feature_columns_all.csv
    label_map.csv
    pydeseq2_cv_results/
      fold_01_deseq.csv
      fold_02_deseq.csv
      ...
    tabnet_cv/
      fold_01/
        selected_feature_columns.csv
        matched_deseq_features.csv
        feature_selection_summary.json
      fold_02/
      ...
      tabnet_cv_metrics.csv
      tabnet_cv_metrics_summary.csv
      tabnet_feature_importance_summary.csv
    baseline_cv/
      svm_linear/
      rf/
      xgboost/
    all_cv_metrics.csv
    all_cv_metrics_summary.csv
```


### 추가된 pretrained TabNet Encoder fine-tuning 옵션

- `TABNET_TRAIN_MODE = "scratch"`: 기존 방식처럼 TabNetClassifier를 처음부터 학습합니다.
- `TABNET_TRAIN_MODE = "pretrained"`: `PRETRAINED_TABNET_RUN_DIR/fold_XX`에서 fold별 `TabNetPretrainer`를 불러와 `from_unsupervised`로 fine-tuning합니다.
- `TABNET_TRAIN_MODE = "both"`: scratch와 pretrained fine-tuning을 같은 CV split에서 모두 수행해 비교합니다.

pretrained fine-tuning을 실행하려면 `PRETRAINED_TABNET_RUN_DIR`에 사전학습 결과 전체 디렉토리를 지정해야 합니다. 각 fold 내부에는 기본적으로 `tabnet_pretrainer_full.zip`, `selected_feature_columns.csv`, 선택적으로 `scaler.joblib`가 있어야 합니다.

DESeq feature selection은 기본값으로 사용하지 않습니다. 모델별 적용 범위는 `DESEQ_FEATURE_SELECTION_SCOPE`로 제어합니다.

- `"none"`: 미사용
- `"all"`: 모든 모델에 적용
- `"tabnet"`: TabNet 계열에 적용
- `"baselines"`: baseline 모델에만 적용
- `"custom"`: `DESEQ_FEATURE_SELECTION_MODELS`에 명시한 모델에만 적용


### pretrained Encoder freeze / unfreeze 옵션

기본값은 Encoder를 freeze하지 않고 함께 fine-tuning하는 방식입니다.

```python
FREEZE_PRETRAINED_ENCODER = False
```

Encoder를 고정하고 classification head 위주로 학습하려면 다음처럼 바꿉니다.

```python
FREEZE_PRETRAINED_ENCODER = True
FREEZE_PRETRAINED_EMBEDDER = True
```

실행 시 `[TabNet trainability]` 로그에 전체 network, encoder, head의 trainable parameter 수가 출력됩니다.
